<a href="https://colab.research.google.com/github/sergiocostaifes/PPCOMP_DM/blob/main/notebooks/AUD02_auditoria_integrada_evidencias.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AUD02 — Auditoria Integrada de Evidências da Linhagem Corrigida — v1.2

**Arquivo:** `AUD02_auditoria_integrada_evidencias.ipynb`  
**Natureza:** governança e auditoria somente-leitura da linhagem experimental vigente.

Execute a célula de código integralmente após a estabilização do pipeline experimental.

A ordem de execução e a dependência de dados são deliberadamente distintas: o NB16_FULL pode ter sido executado como fechamento do ramo, mas **não é consumido pelo AUD02**. A auditoria aceita o protótipo Kaggle refinado, o Parquet FULL materializado e artefatos do ramo FULL somente até o NB15_FULL. Essa barreira evita circularidade entre fechamento experimental e auditoria transversal.

O contrato vigente de atributos é verificado diretamente nas fontes anteriores ao fechamento:

- `raw_core`: 6 atributos;
- `temporal_core`: 13 atributos;
- `event_FAIL_count` preservado no model-facing para proveniência, mas ausente dos preditores;
- ausência de `raw_7` como feature set ativo;
- igualdade semântica `n_failed == event_FAIL_count` no Parquet model-facing;
- ausência de duplicidades exatas injustificadas no vetor efetivamente entregue ao NB11_FULL.

A auditoria não treina modelos, não escolhe hiperparâmetros e não modifica artefatos experimentais.


In [ ]:
# ==============================================================================
# AUD02 — AUDITORIA INTEGRADA DE EVIDÊNCIAS
# Versão 1.2 — auditoria formal da linhagem corrigida, sem dependência do NB16_FULL
#
# Correções desta revisão:
# 1. substituição do lift misto pelo lift alinhado no teste fixo herdado;
# 2. preservação do cálculo misto anterior apenas como legado auditável;
# 3. diagnóstico TSCV complementar, com razão das médias e média das razões;
# 4. check dedicado para AP(score_raw) == AP(score_calibrated);
# 5. distinção entre PASS dos checks e limitações metodológicas em prosa.
# ==============================================================================

# ==============================================================================
# NOTA/SEÇÃO DOCUMENTAL 1
# ==============================================================================
# AUD02 — Auditoria Integrada de Evidências da Linhagem Corrigida
# Notebook: AUD02_auditoria_integrada_evidencias.ipynb
# Natureza: governança e auditoria somente-leitura
# Revisão: v1.2 — auditoria formal da linhagem corrigida, sem dependência do NB16_FULL
# Diretório de saída: 04-reports/AUD02_evidence_audit/
# Objetivo
# Consolidar, reproduzir e rastrear as afirmações quantitativas que poderão ser utilizadas na dissertação, sem reabrir o ciclo experimental.
# O notebook:
# - lê somente artefatos oficiais do protótipo Kaggle refinado e das etapas NB04_FULL a NB15_FULL;
# - não treina modelos, não escolhe hiperparâmetros e não rederiva splits;
# - não altera vencedores ou decisões;
# - não escreve nos diretórios das etapas auditadas;
# - produz artefatos próprios, hashes SHA-256 e manifesto;
# - distingue invariantes estruturais, regressões empíricas e requisitos de rastreabilidade.
# Ordem formal
# text
# Ramo experimental: NB04_FULL–NB15_FULL → NB16_FULL
# Auditoria transversal: AUD02 é executado depois do fechamento experimental, mas audita somente fontes até NB15_FULL.
# O NB16_FULL permanece preservado como inventário do ramo e NÃO é fonte do AUD02; isso elimina circularidade documental.
# O AUD02 é registrado separadamente no inventário documental geral, no Mapa da Escrita e no pacote final da dissertação.


# ==============================================================================
# NOTA/SEÇÃO DOCUMENTAL 2
# ==============================================================================
# Decisões incorporadas
# 2. Três classes de checks:
# - STRUCTURAL: falha indica defeito real e interrompe ao final;
# - REGRESSION: divergência indica mudança a investigar, não erro automático;
# - TRACEABILITY: verifica arquivo, campo, regra e semântica.
# 3. Terminologia factual: usa-se is_modelable=False, não “célula descritiva”.
# 4. Sem inferência indevida:
# - Kaggle × FULL é comparação descritiva entre bases/protocolos distintos;
# - ΔF1/SE ≥ 2 é heuristica_descritiva_nao_pareada_baseada_no_se_da_lstm, nunca teste de significância.
# 5. Lift correto: Lift_PR = PR-AUC / prevalência, com numerador e denominador no mesmo conjunto.
# 6. Snapshot versionado: AUD02_expected_values.json é criado uma vez e nunca sobrescrito automaticamente.
# 7. Congelamento: execução até 31/07/2026 ou exceção explícita para governança somente-leitura.
# 8. Escopo transversal: saída diretamente sob 04-reports, como irmã do ramo _FULL.


# ============================================================
# 0. Bootstrap, montagem do Drive e configurações
# ============================================================

from google.colab import drive
drive.mount("/content/drive", force_remount=False)

from pathlib import Path
from datetime import datetime, date, timezone
from dataclasses import dataclass
from typing import Any, Iterable
import hashlib
import json
import math
import re
import warnings

import numpy as np
import pandas as pd
from sklearn.metrics import average_precision_score
from IPython.display import display, Markdown

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 260)
pd.set_option("display.max_colwidth", 120)

RUN_STARTED_AT = datetime.now(timezone.utc).astimezone()
FREEZE_DATE = date(2026, 7, 31)

DRIVE_ROOT = Path("/content/drive/MyDrive/Mestrado")
REPORTS_DIR = DRIVE_ROOT / "04-reports"
FULL_ROOT = REPORTS_DIR / "99_FULL_downstream"

AUDIT_DIR = REPORTS_DIR / "AUD02_evidence_audit"
AUDIT_DIR.mkdir(parents=True, exist_ok=True)

EXPECTED_VALUES_PATH = AUDIT_DIR / "AUD02_expected_values.json"
GOVERNANCE_EXCEPTION_PATH = AUDIT_DIR / "AUD02_governance_exception.json"

EXPECTED_CELLS = list("abcdefgh")
MIN_POSITIVES_FOR_MODELING = 30
DELTA_F1_THRESHOLD = 0.03
DELTA_F1_SE_RATIO_THRESHOLD = 2.0

STRICT_STRUCTURAL_FAILURE = True

print("RUN_STARTED_AT :", RUN_STARTED_AT.isoformat())
print("DRIVE_ROOT     :", DRIVE_ROOT)
print("FULL_ROOT      :", FULL_ROOT)
print("AUDIT_DIR      :", AUDIT_DIR)
print("FREEZE_DATE    :", FREEZE_DATE.isoformat())

assert DRIVE_ROOT.exists(), f"Raiz do projeto não encontrada: {DRIVE_ROOT}"
assert REPORTS_DIR.exists(), f"Diretório 04-reports não encontrado: {REPORTS_DIR}"
assert FULL_ROOT.exists(), f"Ramo FULL não encontrado: {FULL_ROOT}"


# ============================================================
# 1. Guardas, descoberta e utilitários
# ============================================================

DISCOVERY_LOG: list[dict[str, Any]] = []
SOURCE_REGISTRY: dict[str, Path] = {}
SCHEMA_RESULTS: list[dict[str, Any]] = []

BACKUP_MARKERS = {
    "_backup", "backup", "backups", "99-backups",
    "archive", "historico", "histórico", "old", "antigo"
}

@dataclass(frozen=True)
class ArtifactSpec:
    key: str
    preferred: Path
    filename: str
    required: bool = True
    stage: str = ""
    description: str = ""

def is_under(path: Path, root: Path) -> bool:
    try:
        path.resolve().relative_to(root.resolve())
        return True
    except Exception:
        return False

def safe_output_path(filename: str) -> Path:
    path = (AUDIT_DIR / filename).resolve()
    if not is_under(path, AUDIT_DIR):
        raise RuntimeError(f"Tentativa de escrita fora do AUDIT_DIR: {path}")
    return path

def full_stage_number(path: Path) -> int | None:
    """
    Retorna o número da etapa FULL a partir do primeiro componente
    do caminho relativo a FULL_ROOT.

    Exemplo:
      .../99_FULL_downstream/11_FULL_model_baselines/aggregate/arquivo.csv
      -> 11

    O diretório contêiner 99_FULL_downstream não é interpretado como etapa.
    Arquivos transversais do protótipo refinado, fora de FULL_ROOT,
    retornam None.
    """
    resolved = path.resolve()
    try:
        relative = resolved.relative_to(FULL_ROOT.resolve())
    except ValueError:
        return None

    if not relative.parts:
        return None

    stage_component = relative.parts[0].lower()
    match = re.fullmatch(
        r"(?P<stage>\d{2})(?:[a-z])?_full_.+",
        stage_component,
    )
    return int(match.group("stage")) if match else None


def assert_source_allowed(path: Path) -> None:
    resolved = path.resolve()

    if is_under(resolved, AUDIT_DIR):
        raise RuntimeError(
            f"Saída do AUD02 não pode ser fonte oficial: {resolved}"
        )

    stage_number = full_stage_number(resolved)
    if stage_number is not None and stage_number > 15:
        raise RuntimeError(
            "Circularidade/ordem inválida: o AUD02 aceita somente "
            "artefatos FULL até o NB15. "
            f"Etapa detectada: NB{stage_number:02d}_FULL; fonte: {resolved}"
        )


def valid_search_candidate(path: Path) -> bool:
    resolved = path.resolve()
    text = str(resolved).replace("\\", "/").lower()
    parts = {part.lower() for part in resolved.parts}

    if any(
        marker in parts or f"/{marker}/" in text
        for marker in BACKUP_MARKERS
    ):
        return False

    if is_under(resolved, AUDIT_DIR):
        return False

    stage_number = full_stage_number(resolved)
    if stage_number is not None and stage_number > 15:
        return False

    return True


# Testes unitários mínimos do guardião de etapas.
assert full_stage_number(
    FULL_ROOT / "11_FULL_model_baselines" / "aggregate" / "x.csv"
) == 11
assert full_stage_number(
    FULL_ROOT / "13a_FULL_lstm_tuning" / "aggregate" / "x.csv"
) == 13
assert full_stage_number(
    FULL_ROOT / "15_FULL_visualization" / "aggregate" / "x.csv"
) == 15
assert full_stage_number(
    FULL_ROOT / "16_FULL_inventory" / "aggregate" / "x.csv"
) == 16
assert full_stage_number(
    REPORTS_DIR / "11_winner_model.json"
) is None

def discover_artifact(spec: ArtifactSpec) -> Path | None:
    if spec.preferred.exists() and spec.preferred.is_file():
        path = spec.preferred.resolve()
        assert_source_allowed(path)
        SOURCE_REGISTRY[spec.key] = path
        DISCOVERY_LOG.append({
            "key": spec.key, "status": "FOUND_PREFERRED",
            "path": str(path), "required": spec.required, "stage": spec.stage
        })
        return path

    matches = sorted({
        p.resolve()
        for p in DRIVE_ROOT.rglob(spec.filename)
        if p.is_file() and valid_search_candidate(p)
    }, key=lambda p: (len(p.parts), str(p)))

    if len(matches) == 1:
        path = matches[0]
        assert_source_allowed(path)
        SOURCE_REGISTRY[spec.key] = path
        DISCOVERY_LOG.append({
            "key": spec.key, "status": "FOUND_SEARCH",
            "path": str(path), "required": spec.required, "stage": spec.stage
        })
        return path

    status = "AMBIGUOUS" if matches else "MISSING"
    DISCOVERY_LOG.append({
        "key": spec.key, "status": status,
        "path": " | ".join(map(str, matches)),
        "required": spec.required, "stage": spec.stage
    })
    if spec.required:
        if matches:
            raise RuntimeError(
                f"Artefato ambíguo: {spec.key}\n" +
                "\n".join(f"  - {p}" for p in matches)
            )
        raise FileNotFoundError(
            f"Artefato obrigatório ausente: {spec.key} ({spec.filename})"
        )
    return None

def sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as file_obj:
        for chunk in iter(lambda: file_obj.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()

def fingerprint(path: Path) -> dict[str, Any]:
    stat = path.stat()
    return {
        "path": str(path.resolve()),
        "size_bytes": int(stat.st_size),
        "mtime_ns": int(stat.st_mtime_ns),
        "sha256": sha256_file(path),
    }

def normalize_bool(series: pd.Series) -> pd.Series:
    values = (
        series.astype(str).str.strip().str.lower()
        .str.normalize("NFKD")
        .str.encode("ascii", errors="ignore").str.decode("ascii")
    )
    return values.isin({
        "true", "1", "1.0", "yes", "y", "sim", "s",
        "modelavel", "modelable", "ok"
    })

def to_numeric(df: pd.DataFrame, columns: Iterable[str]) -> pd.DataFrame:
    out = df.copy()
    for col in columns:
        if col in out.columns:
            out[col] = pd.to_numeric(out[col], errors="coerce")
    return out

def first_existing(columns: Iterable[str], candidates: Iterable[str]) -> str | None:
    available = set(columns)
    return next((c for c in candidates if c in available), None)

def require_columns(
    df: pd.DataFrame,
    source_key: str,
    required: Iterable[str],
    optional: Iterable[str] = (),
) -> None:
    required = list(required)
    optional = list(optional)
    missing = [c for c in required if c not in df.columns]
    for col in required:
        SCHEMA_RESULTS.append({
            "source_key": source_key, "column": col,
            "requirement": "required",
            "status": "PASS" if col in df.columns else "FAIL"
        })
    for col in optional:
        SCHEMA_RESULTS.append({
            "source_key": source_key, "column": col,
            "requirement": "optional",
            "status": "PASS" if col in df.columns else "MISSING_OPTIONAL"
        })
    if missing:
        raise ValueError(
            f"Schema inválido em {source_key}; ausentes: {missing}\n"
            f"Disponíveis: {list(df.columns)}"
        )

def read_csv_source(key: str, **kwargs) -> pd.DataFrame:
    return pd.read_csv(
        SOURCE_REGISTRY[key],
        encoding="utf-8-sig",
        low_memory=False,
        **kwargs,
    )

def read_json_source(key: str) -> dict[str, Any]:
    with SOURCE_REGISTRY[key].open("r", encoding="utf-8-sig") as file_obj:
        return json.load(file_obj)

def read_parquet_source(
    key: str,
    columns: list[str] | None = None,
) -> pd.DataFrame:
    return pd.read_parquet(
        SOURCE_REGISTRY[key],
        columns=columns,
    )

def write_csv(df: pd.DataFrame, filename: str) -> Path:
    path = safe_output_path(filename)
    df.to_csv(path, index=False, encoding="utf-8-sig")
    return path

def write_json(payload: Any, filename: str) -> Path:
    path = safe_output_path(filename)
    with path.open("w", encoding="utf-8") as file_obj:
        json.dump(payload, file_obj, ensure_ascii=False, indent=2, default=str)
    return path

def write_text(text: str, filename: str) -> Path:
    path = safe_output_path(filename)
    path.write_text(text, encoding="utf-8")
    return path

def canonical_model_name(value: Any) -> str:
    text = str(value).strip().lower()
    mapping = {
        "logistic regression": "logistic_regression",
        "regressao_logistica": "logistic_regression",
        "histgradientboosting": "hist_gradient_boosting",
        "hist gradient boosting": "hist_gradient_boosting",
        "random forest": "random_forest",
        "extratrees": "extra_trees",
    }
    return mapping.get(text, text)

def source_family(value: Any) -> str | None:
    """
    Normaliza a família da fonte PRIMÁRIA de escores.

    `nb11_primary_lstm_optional_sensitivity` continua significando
    NB11_FULL como fonte primária; LSTM aparece apenas como análise
    opcional de sensibilidade.
    """
    if value is None or pd.isna(value):
        return None

    text = str(value).strip().lower()

    if (
        text.startswith("nb11")
        or "nb11_primary" in text
        or "11_full" in text
        or text in {"baseline", "tabular"}
    ):
        return "NB11_FULL"

    if any(token in text for token in ["nb13a", "13a_full", "tuning"]):
        return "NB13A_FULL"

    if any(token in text for token in ["nb13", "13_full"]):
        return "NB13_FULL"

    if text in {
        "lstm",
        "lstm_primary",
        "lstm_scores",
        "lstm_scores_primary",
    }:
        return "NB13_FULL"

    return text.upper()

def non_null_first(values: Iterable[Any]) -> Any:
    for value in values:
        if value is not None and not pd.isna(value):
            return value
    return np.nan


# ============================================================
# 2. Fontes permitidas: protótipo refinado, dados FULL materializados e artefatos FULL até NB15_FULL
# ============================================================

SPECS = [
    ArtifactSpec(
        "refined_winner",
        REPORTS_DIR / "11_winner_model.json",
        "11_winner_model.json",
        True, "PROTO_REFINADO",
        "Vencedor do protótipo Kaggle refinado",
    ),
    ArtifactSpec(
        "refined_metrics_summary",
        REPORTS_DIR / "11_metrics_summary.csv",
        "11_metrics_summary.csv",
        False, "PROTO_REFINADO",
        "Resumo de métricas do protótipo refinado",
    ),
    ArtifactSpec(
        "full_model_facing",
        DRIVE_ROOT / "02-datasets" / "99-full" / "03-model-facing" /
        "window_5min_series_allcells_model_facing_000000000000.parquet",
        "window_5min_series_allcells_model_facing_000000000000.parquet",
        True, "FULL_DATASET",
        "Parquet model-facing FULL usado para verificar a semântica n_failed/event_FAIL_count",
    ),
    ArtifactSpec(
        "full_metrics_summary",
        FULL_ROOT / "11_FULL_model_baselines" / "aggregate" /
        "11_FULL_metrics_summary_by_cell.csv",
        "11_FULL_metrics_summary_by_cell.csv",
        True, "NB11_FULL",
        "Métricas por célula/cenário/modelo",
    ),
    ArtifactSpec(
        "full_metrics_tscv",
        FULL_ROOT / "11_FULL_model_baselines" / "aggregate" /
        "11_FULL_metrics_tscv_by_cell.csv",
        "11_FULL_metrics_tscv_by_cell.csv",
        True, "NB11_FULL",
        "Métricas brutas por fold para diagnóstico TSCV do lift",
    ),
    ArtifactSpec(
        "full_scores_calibrated",
        FULL_ROOT / "11_FULL_model_baselines" / "aggregate" /
        "11_FULL_scores_calibrated_by_cell.parquet",
        "11_FULL_scores_calibrated_by_cell.parquet",
        True, "NB11_FULL",
        "Escores congelados do teste fixo herdado",
    ),
    ArtifactSpec(
        "full_modelability_overview",
        FULL_ROOT / "11_FULL_model_baselines" / "aggregate" /
        "11_FULL_modelability_cell_overview.csv",
        "11_FULL_modelability_cell_overview.csv",
        True, "NB11_FULL",
        "Gate de modelabilidade por célula",
    ),
    ArtifactSpec(
        "full_feature_sets",
        FULL_ROOT / "11_FULL_model_baselines" / "aggregate" /
        "11_FULL_feature_sets.json",
        "11_FULL_feature_sets.json",
        True, "NB11_FULL",
        "Contrato dos feature sets ativos do NB11_FULL",
    ),
    ArtifactSpec(
        "full_model_input_features",
        FULL_ROOT / "11_FULL_model_baselines" / "aggregate" /
        "11_FULL_model_input_features_by_cell.parquet",
        "11_FULL_model_input_features_by_cell.parquet",
        True, "NB11_FULL",
        "Matriz agregada de atributos efetivamente entregue aos modelos tabulares",
    ),
    ArtifactSpec(
        "nb13_decision",
        FULL_ROOT / "13_FULL_lstm" / "aggregate" /
        "13_FULL_lstm_decision_to_nb14_by_cell.csv",
        "13_FULL_lstm_decision_to_nb14_by_cell.csv",
        False, "NB13_FULL",
        "Decisão LSTM base para o NB14",
    ),
    ArtifactSpec(
        "nb13_uncertainty",
        FULL_ROOT / "13_FULL_lstm" / "aggregate" /
        "13_FULL_lstm_delta_uncertainty_by_cell.csv",
        "13_FULL_lstm_delta_uncertainty_by_cell.csv",
        False, "NB13_FULL",
        "Delta e incerteza descritiva não pareada",
    ),
    ArtifactSpec(
        "nb13a_decision",
        FULL_ROOT / "13a_FULL_lstm_tuning" / "aggregate" /
        "13a_FULL_lstm_tuning_decision_to_nb14_by_cell.csv",
        "13a_FULL_lstm_tuning_decision_to_nb14_by_cell.csv",
        False, "NB13A_FULL",
        "Decisão soberana após a varredura LSTM",
    ),
    ArtifactSpec(
        "nb13a_delta_nb11",
        FULL_ROOT / "13a_FULL_lstm_tuning" / "aggregate" /
        "13a_FULL_lstm_tuning_delta_vs_NB11_FULL_by_cell.csv",
        "13a_FULL_lstm_tuning_delta_vs_NB11_FULL_by_cell.csv",
        False, "NB13A_FULL",
        "Delta da LSTM ajustada contra NB11_FULL",
    ),
    ArtifactSpec(
        "nb14_scenario_summary",
        FULL_ROOT / "14_FULL_decision_analysis" / "aggregate" /
        "14_FULL_scenario_summary_by_cell.csv",
        "14_FULL_scenario_summary_by_cell.csv",
        True, "NB14_FULL",
        "Resumo operacional por célula/cenário/fonte",
    ),
    ArtifactSpec(
        "nb14_episode_by_tau",
        FULL_ROOT / "14_FULL_decision_analysis" / "aggregate" /
        "14_FULL_episode_anticipability_by_tau_allcells.csv",
        "14_FULL_episode_anticipability_by_tau_allcells.csv",
        False, "NB14_FULL",
        "Antecipação por episódio e tau",
    ),
    ArtifactSpec(
        "nb14_episode_all",
        FULL_ROOT / "14_FULL_decision_analysis" / "aggregate" /
        "14_FULL_episode_anticipability_allcells.csv",
        "14_FULL_episode_anticipability_allcells.csv",
        False, "NB14_FULL",
        "Detalhe por episódio",
    ),
]

# Guarda declarativa adicional: uma mudança futura em SPECS não pode
# introduzir silenciosamente NB16_FULL ou qualquer etapa posterior.
for _spec in SPECS:
    _stage = full_stage_number(_spec.preferred)
    if _stage is not None and _stage > 15:
        raise RuntimeError(
            f"Fonte inválida configurada em SPECS: {_spec.key} -> "
            f"NB{_stage:02d}_FULL. AUD02 audita somente até NB15_FULL."
        )
    if str(_spec.stage).upper().startswith("NB16"):
        raise RuntimeError(
            f"Fonte NB16_FULL proibida no AUD02: {_spec.key}"
        )


for spec in SPECS:
    discover_artifact(spec)

discovery_df = pd.DataFrame(DISCOVERY_LOG)
display(discovery_df)

SOURCE_FINGERPRINTS_BEFORE = {
    key: fingerprint(path)
    for key, path in SOURCE_REGISTRY.items()
}


# ==============================================================================
# NOTA/SEÇÃO DOCUMENTAL 3
# ==============================================================================
# Módulo 7 — Comparação descritiva com o valor de referência do protótipo refinado
# Este módulo é executado primeiro porque sua lógica já foi validada no item 4.9.
# A comparação:
# - seleciona um resultado soberano por célula com base em F1, recall e Average Precision;
# - compara o ROC-AUC desse resultado com o valor de referência;
# - não escolhe modelos por ROC-AUC;
# - não realiza teste inferencial entre bases;
# - registra dispersão entre folds apenas como descrição.


# ============================================================
# Módulo 7 — lógica do item 4.9
# ============================================================

refined_winner = read_json_source("refined_winner")
benchmark_field = first_existing(
    refined_winner.keys(),
    ["roc_auc_tscv_mean", "roc_auc_mean", "mean_roc_auc", "roc_auc"],
)
if benchmark_field is None:
    raise KeyError(
        "JSON do vencedor refinado sem campo reconhecido de ROC-AUC. "
        f"Campos: {sorted(refined_winner)}"
    )
KAGGLE_REFERENCE_ROC_AUC = float(refined_winner[benchmark_field])

full_summary = read_csv_source("full_metrics_summary")
require_columns(
    full_summary,
    "full_metrics_summary",
    required=[
        "cell_id", "scenario_label", "feature_set", "model",
        "f1_tscv_mean", "recall_tscv_mean",
        "average_precision_tscv_mean", "roc_auc_tscv_mean",
    ],
    optional=[
        "f1_tscv_std", "roc_auc_tscv_std",
        "status_modelagem", "is_modelable",
        "n_positivos_train", "n_positivos_test",
        "brier_tscv_mean", "precision_tscv_mean",
    ],
)

full_summary = to_numeric(
    full_summary,
    [
        "f1_tscv_mean", "recall_tscv_mean",
        "average_precision_tscv_mean", "roc_auc_tscv_mean",
        "f1_tscv_std", "roc_auc_tscv_std",
        "n_positivos_train", "n_positivos_test",
        "brier_tscv_mean", "precision_tscv_mean",
    ],
)
full_summary["cell_id"] = (
    full_summary["cell_id"].astype(str).str.strip().str.lower()
)
full_summary["model"] = full_summary["model"].map(canonical_model_name)

if "is_modelable" in full_summary.columns:
    full_summary["is_modelable_normalized"] = normalize_bool(
        full_summary["is_modelable"]
    )
elif "status_modelagem" in full_summary.columns:
    full_summary["is_modelable_normalized"] = normalize_bool(
        full_summary["status_modelagem"]
    )
else:
    raise ValueError(
        "Resumo NB11_FULL sem is_modelable ou status_modelagem."
    )

modelable_summary = full_summary[
    full_summary["is_modelable_normalized"]
].copy()

sovereign = (
    modelable_summary
    .sort_values(
        [
            "cell_id", "f1_tscv_mean", "recall_tscv_mean",
            "average_precision_tscv_mean",
        ],
        ascending=[True, False, False, False],
        kind="mergesort",
    )
    .groupby("cell_id", sort=True, group_keys=False)
    .head(1)
    .copy()
)

sovereign["benchmark_roc_auc"] = KAGGLE_REFERENCE_ROC_AUC
sovereign["delta_roc_auc"] = (
    sovereign["roc_auc_tscv_mean"] - KAGGLE_REFERENCE_ROC_AUC
)
sovereign["supera_referencia"] = (
    sovereign["roc_auc_tscv_mean"] > KAGGLE_REFERENCE_ROC_AUC
)
sovereign["comparison_semantics"] = (
    "comparacao_descritiva_entre_bases_protocolos_e_cenarios_nao_identicos"
)
sovereign["inferential_test"] = "nao_aplicavel"

# Estatísticas descritivas dos folds.
#
# O resumo oficial pode já conter n_folds_valid ou alguma das colunas
# abaixo. Antes do merge, essas colunas são removidas para impedir a
# criação automática de sufixos _x/_y pelo pandas. Quando o arquivo
# bruto TSCV existe, ele é a fonte preferencial para média, dispersão,
# mínimo, máximo e número de folds válidos.
fold_stat_columns = [
    "roc_auc_fold_mean",
    "roc_auc_fold_std",
    "roc_auc_fold_min",
    "roc_auc_fold_max",
    "n_folds_valid",
]

if "full_metrics_tscv" in SOURCE_REGISTRY:
    raw_tscv = read_csv_source("full_metrics_tscv")
    require_columns(
        raw_tscv,
        "full_metrics_tscv",
        required=[
            "cell_id", "scenario_label", "feature_set",
            "model", "fold", "roc_auc",
        ],
        optional=["f1", "average_precision", "is_modelable"],
    )
    raw_tscv["cell_id"] = (
        raw_tscv["cell_id"].astype(str).str.strip().str.lower()
    )
    raw_tscv["model"] = raw_tscv["model"].map(canonical_model_name)
    raw_tscv = to_numeric(raw_tscv, ["fold", "roc_auc"])

    roc_fold_stats = (
        raw_tscv
        .groupby(
            ["cell_id", "scenario_label", "feature_set", "model"],
            as_index=False,
            dropna=False,
        )
        .agg(
            roc_auc_fold_mean=("roc_auc", "mean"),
            roc_auc_fold_std=("roc_auc", "std"),
            roc_auc_fold_min=("roc_auc", "min"),
            roc_auc_fold_max=("roc_auc", "max"),
            n_folds_valid=("fold", "nunique"),
        )
    )

    # Evita colisões como n_folds_valid_x/n_folds_valid_y.
    sovereign = sovereign.drop(
        columns=[
            column
            for column in fold_stat_columns
            if column in sovereign.columns
        ],
        errors="ignore",
    )

    sovereign = sovereign.merge(
        roc_fold_stats,
        on=["cell_id", "scenario_label", "feature_set", "model"],
        how="left",
        validate="one_to_one",
    )
else:
    if "roc_auc_fold_mean" not in sovereign.columns:
        sovereign["roc_auc_fold_mean"] = sovereign[
            "roc_auc_tscv_mean"
        ]
    if "roc_auc_fold_std" not in sovereign.columns:
        sovereign["roc_auc_fold_std"] = sovereign.get(
            "roc_auc_tscv_std",
            np.nan,
        )
    if "roc_auc_fold_min" not in sovereign.columns:
        sovereign["roc_auc_fold_min"] = np.nan
    if "roc_auc_fold_max" not in sovereign.columns:
        sovereign["roc_auc_fold_max"] = np.nan
    if "n_folds_valid" not in sovereign.columns:
        sovereign["n_folds_valid"] = np.nan

# Defesa adicional: assegura o schema antes da seleção das colunas.
for column in fold_stat_columns:
    if column not in sovereign.columns:
        sovereign[column] = np.nan

comparison_columns = [
    "cell_id", "scenario_label", "feature_set", "model",
    "f1_tscv_mean", "recall_tscv_mean",
    "average_precision_tscv_mean", "roc_auc_tscv_mean",
    "roc_auc_fold_std", "roc_auc_fold_min", "roc_auc_fold_max",
    "n_folds_valid", "benchmark_roc_auc", "delta_roc_auc",
    "supera_referencia", "comparison_semantics", "inferential_test",
]
for col in [
    "status_modelagem", "is_modelable",
    "n_positivos_train", "n_positivos_test",
]:
    if col in sovereign.columns:
        comparison_columns.append(col)

kaggle_comparison = (
    sovereign[comparison_columns]
    .sort_values("cell_id")
    .reset_index(drop=True)
)

PATH_KAGGLE_COMPARISON = write_csv(
    kaggle_comparison,
    "AUD02_kaggle_comparison.csv",
)

display(
    kaggle_comparison.style.format({
        "f1_tscv_mean": "{:.6f}",
        "roc_auc_tscv_mean": "{:.6f}",
        "roc_auc_fold_std": "{:.6f}",
        "benchmark_roc_auc": "{:.6f}",
        "delta_roc_auc": "{:+.6f}",
    })
)

print("Valor de referência:", f"{KAGGLE_REFERENCE_ROC_AUC:.12f}")
print(
    "Acima:",
    kaggle_comparison.loc[
        kaggle_comparison["supera_referencia"], "cell_id"
    ].tolist(),
)
print(
    "Não acima:",
    kaggle_comparison.loc[
        ~kaggle_comparison["supera_referencia"], "cell_id"
    ].tolist(),
)


# ==============================================================================
# NOTA/SEÇÃO DOCUMENTAL 4
# ==============================================================================
# Módulo 9 — Matriz de afirmações como especificação
# A matriz é definida antes dos demais módulos. Cada módulo existe para fechar uma afirmação já destinada à dissertação ou à governança.


# ============================================================
# Módulo 9 — especificação inicial
# ============================================================

claims_spec = pd.DataFrame([
    {
        "claim_id": "4.1",
        "claim_text": (
            "Existem oito células; sete têm is_modelable=True e "
            "a única com is_modelable=False é d."
        ),
        "module": "M3",
        "required_evidence": "NB11_FULL modelability + suporte por cenário",
        "derivation_rule_spec": "contagem de cell_id e filtro por is_modelable",
        "target_status": "CONFIRMED",
    },
    {
        "claim_id": "4.2",
        "claim_text": (
            "O ROC-AUC soberano das células modeláveis varia "
            "aproximadamente de 0,637 a 0,813."
        ),
        "module": "M7",
        "required_evidence": "NB11_FULL metrics summary",
        "derivation_rule_spec": (
            "min/max após seleção soberana por F1/recall/AP"
        ),
        "target_status": "CONFIRMED",
    },
    {
        "claim_id": "4.3",
        "claim_text": (
            "No teste fixo herdado, o lift de PR-AUC sobre "
            "prevalência é calculado com numerador e denominador "
            "sobre exatamente as mesmas observações."
        ),
        "module": "M6",
        "required_evidence": (
            "escores NB11_FULL congelados + seleção primária NB14_FULL"
        ),
        "derivation_rule_spec": (
            "average_precision_score(y_true, score_calibrated) "
            "/ mean(y_true)"
        ),
        "target_status": "CONFIRMED",
    },
    {
        "claim_id": "4.4",
        "claim_text": (
            "A mediana da taxa de episódios antecipados no limiar "
            "de referência é aproximadamente 33%."
        ),
        "module": "M8",
        "required_evidence": "NB14_FULL: tau, numerador e denominador",
        "derivation_rule_spec": (
            "mediana entre células modeláveis de antecipados/avaliáveis"
        ),
        "target_status": "CONFIRMED",
    },
    {
        "claim_id": "4.5",
        "claim_text": (
            "A Regressão Logística é o vencedor do protótipo "
            "Kaggle refinado no cenário de referência."
        ),
        "module": "M7/M9",
        "required_evidence": "11_winner_model.json",
        "derivation_rule_spec": "leitura direta do campo model",
        "target_status": "CONFIRMED",
    },
    {
        "claim_id": "4.6",
        "claim_text": (
            "No FULL, o resultado soberano não é dominado "
            "exclusivamente pela Regressão Logística."
        ),
        "module": "M4/M5",
        "required_evidence": "ranking NB11_FULL + governança NB13/NB14",
        "derivation_rule_spec": "frequência e fonte efetiva de escores",
        "target_status": "CONFIRMED",
    },
    {
        "claim_id": "4.7",
        "claim_text": (
            "Nenhuma LSTM foi promovida como fonte primária. "
            "As células que ultrapassam o piso nominal de ΔF1 são "
            "determinadas a partir dos artefatos vigentes; o gate de "
            "robustez e a promoção efetiva são auditados separadamente. "
            "As LSTMs do NB13_FULL e NB13a_FULL permanecem análises "
            "secundárias quando não promovidas."
        ),
        "module": "M5",
        "required_evidence": "NB13/NB13a/NB14",
        "derivation_rule_spec": "cruzamento por cell_id",
        "target_status": "CONFIRMED_OR_QUALIFIED",
    },
    {
        "claim_id": "4.8",
        "claim_text": (
            "Brier/calibração são contexto e limitação, "
            "sem reabrir o escopo."
        ),
        "module": "M4/M9",
        "required_evidence": "métricas existentes e decisão documental",
        "derivation_rule_spec": "registro sem nova otimização",
        "target_status": "CONTEXT_ONLY",
    },
    {
        "claim_id": "4.9",
        "claim_text": (
            "Seis das sete células modeláveis superam descritivamente "
            "a referência; f fica abaixo."
        ),
        "module": "M7",
        "required_evidence": "AUD02_kaggle_comparison.csv",
        "derivation_rule_spec": "roc_auc_tscv_mean > referência",
        "target_status": "CONFIRMED",
    },
])

display(claims_spec)


# ==============================================================================
# NOTA/SEÇÃO DOCUMENTAL 5
# ==============================================================================
# Módulo 2 — Taxonomia de baselines e modelos
# A taxonomia evita que “baseline” seja usado indistintamente para regra trivial, modelo supervisionado tabular e LSTM.


# ============================================================
# Módulo 2 — taxonomia
# ============================================================

model_taxonomy = pd.DataFrame([
    {
        "component": "always_negative",
        "category": "baseline_trivial",
        "role": "referencia_de_nao_acionamento",
        "supervised_training": False,
        "promotion_candidate": False,
        "notes": "Prediz sempre a classe negativa.",
    },
    {
        "component": "fixed_cut_rule",
        "category": "baseline_temporal_regra",
        "role": "regra_interpretavel_de_acionamento",
        "supervised_training": False,
        "promotion_candidate": False,
        "notes": "Regra baseada em limiar fixo.",
    },
    {
        "component": "ewma_causal",
        "category": "baseline_temporal_regra",
        "role": "baseline_temporal_competitivo",
        "supervised_training": False,
        "promotion_candidate": False,
        "notes": "EWMA causal; não utiliza observações futuras.",
    },
    {
        "component": "logistic_regression",
        "category": "modelo_supervisionado_tabular",
        "role": "candidato_tabular",
        "supervised_training": True,
        "promotion_candidate": True,
        "notes": "Modelo linear interpretável.",
    },
    {
        "component": "random_forest",
        "category": "modelo_supervisionado_tabular",
        "role": "candidato_tabular",
        "supervised_training": True,
        "promotion_candidate": True,
        "notes": "Ensemble por bagging.",
    },
    {
        "component": "hist_gradient_boosting",
        "category": "modelo_supervisionado_tabular",
        "role": "candidato_tabular",
        "supervised_training": True,
        "promotion_candidate": True,
        "notes": "Ensemble por boosting.",
    },
    {
        "component": "extra_trees",
        "category": "modelo_supervisionado_tabular",
        "role": "candidato_tabular_quando_presente",
        "supervised_training": True,
        "promotion_candidate": True,
        "notes": "Ensemble de árvores extremamente aleatorizadas.",
    },
    {
        "component": "lstm",
        "category": "modelo_sequencial_comparador",
        "role": "candidata_a_promocao_sob_gates",
        "supervised_training": True,
        "promotion_candidate": True,
        "notes": "Não é baseline principal; depende de governança.",
    },
])

PATH_MODEL_TAXONOMY = write_csv(
    model_taxonomy,
    "AUD02_model_taxonomy.csv",
)
display(model_taxonomy)


# ==============================================================================
# NOTA/SEÇÃO DOCUMENTAL 6
# ==============================================================================
# Módulo 3 — Modelabilidade
# O módulo diferencia:
# - unidade janela: positivos de treino e teste;
# - unidade episódio: episódios totais e avaliáveis;
# - campo factual: is_modelable.
# A expressão “célula descritiva” não substitui o campo.


# ============================================================
# Módulo 3 — modelabilidade
# ============================================================

modelability_overview_raw = read_csv_source(
    "full_modelability_overview"
)
require_columns(
    modelability_overview_raw,
    "full_modelability_overview",
    required=["cell_id"],
    optional=[
        "is_modelable", "status_modelagem",
        "n_modelable_scenarios", "n_nonmodelable_scenarios",
        "n_positivos_train", "n_positivos_test",
    ],
)

modelability_overview_raw["cell_id"] = (
    modelability_overview_raw["cell_id"]
    .astype(str).str.strip().str.lower()
)

scenario_cols = [
    "cell_id", "scenario_label", "is_modelable_normalized",
    "status_modelagem", "is_modelable",
    "n_positivos_train", "n_positivos_test",
]
available_scenario_cols = [
    c for c in scenario_cols if c in full_summary.columns
]

modelability_by_scenario = (
    full_summary[available_scenario_cols]
    .drop_duplicates()
    .sort_values(["cell_id", "scenario_label"])
    .reset_index(drop=True)
)

if "is_modelable_normalized" not in modelability_by_scenario.columns:
    if "is_modelable" in modelability_by_scenario.columns:
        modelability_by_scenario["is_modelable_normalized"] = (
            normalize_bool(modelability_by_scenario["is_modelable"])
        )
    elif "status_modelagem" in modelability_by_scenario.columns:
        modelability_by_scenario["is_modelable_normalized"] = (
            normalize_bool(modelability_by_scenario["status_modelagem"])
        )

agg_spec = {
    "n_scenarios": ("scenario_label", "nunique"),
    "n_modelable_scenarios": (
        "is_modelable_normalized", "sum"
    ),
    "is_modelable": ("is_modelable_normalized", "max"),
}
if "n_positivos_train" in modelability_by_scenario.columns:
    agg_spec["min_n_positivos_train"] = (
        "n_positivos_train", "min"
    )
if "n_positivos_test" in modelability_by_scenario.columns:
    agg_spec["min_n_positivos_test"] = (
        "n_positivos_test", "min"
    )

modelability_summary = (
    modelability_by_scenario
    .groupby("cell_id", as_index=False)
    .agg(**agg_spec)
)

overview_extra = (
    modelability_overview_raw
    .drop_duplicates("cell_id")
)
modelability_summary = modelability_summary.merge(
    overview_extra,
    on="cell_id",
    how="left",
    suffixes=("", "_overview"),
    validate="one_to_one",
)

nb14_summary = read_csv_source("nb14_scenario_summary")
require_columns(
    nb14_summary,
    "nb14_scenario_summary",
    required=[
        "cell_id", "scenario_label", "score_source",
        "is_primary_score_source", "n_total_episodes",
        "n_evaluable_episodes_tau_reference",
        "n_scored_rows", "positive_rate_tau_reference",
        "model", "feature_set", "score_source_file",
    ],
    optional=[
        "tau_reference",
        "n_anticipated_episodes_tau_reference",
        "episode_anticipation_rate_tau_reference",
    ],
)
nb14_summary["cell_id"] = (
    nb14_summary["cell_id"].astype(str).str.strip().str.lower()
)
nb14_summary["is_primary_normalized"] = normalize_bool(
    nb14_summary["is_primary_score_source"]
)
nb14_primary = (
    nb14_summary[nb14_summary["is_primary_normalized"]]
    .sort_values(["cell_id", "scenario_label"])
    .drop_duplicates("cell_id")
)

episode_support_cols = [
    c for c in [
        "cell_id", "scenario_label", "n_total_episodes",
        "n_evaluable_episodes_tau_reference", "tau_reference",
    ]
    if c in nb14_primary.columns
]
episode_support = nb14_primary[episode_support_cols].copy()

modelability_summary = modelability_summary.merge(
    episode_support,
    on="cell_id",
    how="left",
    validate="one_to_one",
)

PATH_MODELABILITY_SCENARIO = write_csv(
    modelability_by_scenario,
    "AUD02_modelability_by_cell_scenario.csv",
)
PATH_MODELABILITY_SUMMARY = write_csv(
    modelability_summary,
    "AUD02_modelability_summary.csv",
)

display(modelability_summary.sort_values("cell_id"))


# ==============================================================================
# NOTA/SEÇÃO DOCUMENTAL 7
# ==============================================================================
# Módulo 4 — Ranking e seleção dos modelos tabulares
# São produzidos:
# 1. ranking por célula, cenário e conjunto de atributos;
# 2. resultado soberano por célula;
# 3. frequência empírica por família de modelo.


# ============================================================
# Módulo 4 — ranking e seleção
# ============================================================

ranking = modelable_summary.copy()

ranking = ranking.sort_values(
    [
        "cell_id", "scenario_label", "feature_set",
        "f1_tscv_mean", "recall_tscv_mean",
        "average_precision_tscv_mean",
    ],
    ascending=[True, True, True, False, False, False],
    kind="mergesort",
)

ranking["rank_within_cell_scenario_feature_set"] = (
    ranking
    .groupby(["cell_id", "scenario_label", "feature_set"])
    .cumcount()
    + 1
)
ranking["is_nominal_winner_within_group"] = (
    ranking["rank_within_cell_scenario_feature_set"] == 1
)

sovereign_keys = sovereign[
    ["cell_id", "scenario_label", "feature_set", "model"]
].copy()
sovereign_keys["is_sovereign_winner"] = True

ranking = ranking.merge(
    sovereign_keys,
    on=["cell_id", "scenario_label", "feature_set", "model"],
    how="left",
)
ranking["is_sovereign_winner"] = (
    ranking["is_sovereign_winner"].astype("boolean").fillna(False).astype(bool)
)

nominal_winners = (
    ranking[ranking["is_sovereign_winner"]]
    .copy()
    .sort_values("cell_id")
    .reset_index(drop=True)
)

model_frequency = (
    nominal_winners
    .groupby("model", as_index=False)
    .agg(n_cells=("cell_id", "nunique"))
    .sort_values(["n_cells", "model"], ascending=[False, True])
    .reset_index(drop=True)
)

PATH_BASELINE_RANKING = write_csv(
    ranking,
    "AUD02_baseline_ranking_by_cell_scenario.csv",
)
PATH_NOMINAL_WINNERS = write_csv(
    nominal_winners,
    "AUD02_nominal_winners_by_cell.csv",
)
PATH_MODEL_FREQUENCY = write_csv(
    model_frequency,
    "AUD02_model_frequency.csv",
)

display(model_frequency)
display(
    nominal_winners[
        [
            "cell_id", "scenario_label", "feature_set",
            "model", "f1_tscv_mean", "roc_auc_tscv_mean",
            "average_precision_tscv_mean",
        ]
    ]
)


# ==============================================================================
# NOTA/SEÇÃO DOCUMENTAL 8
# ==============================================================================
# Módulo 5 — Governança do modelo efetivamente utilizado
# A razão ΔF1/SE é uma heurística descritiva não pareada, baseada no erro-padrão da F1 da LSTM.
# Ela não é interpretada como:
# - p-valor;
# - significância estatística;
# - intervalo de confiança;
# - evidência transferível para Kaggle × FULL.


# ============================================================
# Módulo 5 — governança NB11 → NB13/NB13a → NB14
# ============================================================

OPTIONAL_SOURCE_FRAMES: dict[str, pd.DataFrame] = {}

for key in [
    "nb13_decision",
    "nb13_uncertainty",
    "nb13a_decision",
    "nb13a_delta_nb11",
]:
    if key in SOURCE_REGISTRY:
        frame = read_csv_source(key)
        if "cell_id" in frame.columns:
            frame["cell_id"] = (
                frame["cell_id"]
                .astype(str)
                .str.strip()
                .str.lower()
            )
        OPTIONAL_SOURCE_FRAMES[key] = frame


def value_from_source(
    source_key: str,
    cell_id: str,
    candidates: list[str],
) -> tuple[Any, str | None]:
    frame = OPTIONAL_SOURCE_FRAMES.get(source_key)

    if frame is None or "cell_id" not in frame.columns:
        return np.nan, None

    rows = frame[frame["cell_id"] == cell_id]
    if rows.empty:
        return np.nan, None

    for column in candidates:
        if column not in rows.columns:
            continue

        value = non_null_first(rows[column].tolist())
        if value is not None and not pd.isna(value):
            return value, column

    return np.nan, None


def numeric_value(value: Any) -> float:
    converted = pd.to_numeric(
        pd.Series([value]),
        errors="coerce",
    ).iloc[0]
    return float(converted) if not pd.isna(converted) else np.nan


LSTM_F1_ALIASES = [
    "best_lstm_tuned_f1_tscv_mean",
    "best_lstm_f1_tscv_mean",
    "lstm_f1_tscv_mean",
    "tuned_lstm_f1_tscv_mean",
    "lstm_f1_mean",
    "f1_lstm",
    "lstm_mean_f1",
    "candidate_f1",
    "challenger_f1",
]

DELTA_F1_ALIASES = [
    "delta_f1_tuned_minus_nb11",
    "delta_f1_lstm_minus_nb11",
    "delta_f1",
    "delta_f1_mean",
    "f1_delta",
    "delta_vs_nb11",
    "delta_vs_baseline",
]

LSTM_F1_SE_ALIASES = [
    "best_lstm_tuned_f1_se",
    "best_lstm_f1_se",
    "lstm_f1_se",
    "f1_se",
]

RATIO_ALIASES = [
    "delta_over_lstm_tuned_f1_se",
    "delta_over_lstm_f1_se",
    "delta_f1_se_ratio",
    "delta_over_se",
    "delta_f1_over_se",
    "signal_to_noise_ratio",
]

DECISION_ALIASES = [
    "final_position",
    "governance_decision",
    "decision",
    "recommendation",
    "final_decision",
]

SCORE_RECOMMENDATION_ALIASES = [
    "nb14_score_source_recommendation",
    "score_source_for_nb14",
    "recommended_score_source",
    "score_source_recommendation",
    "final_score_source",
    "score_source",
]

governance_rows = []

for _, base_row in nominal_winners.sort_values("cell_id").iterrows():
    cell_id = base_row["cell_id"]

    # Resultado LSTM: NB13a_FULL tem prioridade.
    lstm_f1 = np.nan
    lstm_f1_field = None
    lstm_f1_source = None

    for source_key in [
        "nb13a_decision",
        "nb13a_delta_nb11",
        "nb13_uncertainty",
        "nb13_decision",
    ]:
        value, field = value_from_source(
            source_key,
            cell_id,
            LSTM_F1_ALIASES,
        )
        if field is not None:
            lstm_f1 = numeric_value(value)
            lstm_f1_field = field
            lstm_f1_source = source_key
            break

    # Delta contra o vencedor NB11_FULL.
    delta_f1 = np.nan
    delta_field = None
    delta_source = None

    for source_key in [
        "nb13a_decision",
        "nb13a_delta_nb11",
        "nb13_uncertainty",
        "nb13_decision",
    ]:
        value, field = value_from_source(
            source_key,
            cell_id,
            DELTA_F1_ALIASES,
        )
        if field is not None:
            delta_f1 = numeric_value(value)
            delta_field = field
            delta_source = source_key
            break

    if pd.isna(delta_f1) and not pd.isna(lstm_f1):
        delta_f1 = (
            float(lstm_f1)
            - float(base_row["f1_tscv_mean"])
        )
        delta_field = "derived:lstm_f1-baseline_f1"
        delta_source = lstm_f1_source

    # Erro-padrão da F1 da LSTM, não da diferença pareada.
    lstm_f1_se = np.nan
    lstm_f1_se_field = None
    lstm_f1_se_source = None

    for source_key in [
        "nb13a_decision",
        "nb13_uncertainty",
        "nb13_decision",
    ]:
        value, field = value_from_source(
            source_key,
            cell_id,
            LSTM_F1_SE_ALIASES,
        )
        if field is not None:
            lstm_f1_se = numeric_value(value)
            lstm_f1_se_field = field
            lstm_f1_se_source = source_key
            break

    # Heurística descritiva delta / SE da LSTM.
    ratio = np.nan
    ratio_field = None
    ratio_source = None

    for source_key in [
        "nb13a_decision",
        "nb13_uncertainty",
        "nb13_decision",
    ]:
        value, field = value_from_source(
            source_key,
            cell_id,
            RATIO_ALIASES,
        )
        if field is not None:
            ratio = numeric_value(value)
            ratio_field = field
            ratio_source = source_key
            break

    if (
        pd.isna(ratio)
        and not pd.isna(delta_f1)
        and not pd.isna(lstm_f1_se)
        and float(lstm_f1_se) != 0.0
    ):
        ratio = float(delta_f1) / float(lstm_f1_se)
        ratio_field = "derived:delta_f1/lstm_f1_se"
        ratio_source = delta_source or lstm_f1_se_source

    # Decisão nominal.
    decision = np.nan
    decision_field = None
    decision_source = None

    for source_key in ["nb13a_decision", "nb13_decision"]:
        value, field = value_from_source(
            source_key,
            cell_id,
            DECISION_ALIASES,
        )
        if field is not None:
            decision = value
            decision_field = field
            decision_source = source_key
            break

    # Recomendação soberana da fonte primária para o NB14_FULL.
    recommended_score_source = np.nan
    recommended_source_field = None
    recommended_source_evidence = None

    for source_key in ["nb13a_decision", "nb13_decision"]:
        value, field = value_from_source(
            source_key,
            cell_id,
            SCORE_RECOMMENDATION_ALIASES,
        )
        if field is not None:
            recommended_score_source = value
            recommended_source_field = field
            recommended_source_evidence = source_key
            break

    # Fonte efetivamente utilizada pelo NB14_FULL.
    nb14_row = nb14_primary[
        nb14_primary["cell_id"] == cell_id
    ]

    if nb14_row.empty:
        actual_score_source = np.nan
        actual_score_file = np.nan
    else:
        actual_score_source = non_null_first(
            nb14_row["score_source"].tolist()
        )
        actual_score_file = (
            non_null_first(
                nb14_row["score_source_file"].tolist()
            )
            if "score_source_file" in nb14_row.columns
            else np.nan
        )

    recommended_family = source_family(
        recommended_score_source
    )
    actual_family = source_family(
        actual_score_source
    )

    if recommended_family is None:
        score_source_coherence = (
            "UNRESOLVED_RECOMMENDATION"
        )
    elif actual_family is None:
        score_source_coherence = "UNRESOLVED_ACTUAL"
    elif recommended_family == actual_family:
        score_source_coherence = "COHERENT"
    else:
        score_source_coherence = "REVIEW"

    governance_rows.append({
        "cell_id": cell_id,
        "baseline_nominal_winner": base_row["model"],
        "baseline_scenario": base_row["scenario_label"],
        "baseline_feature_set": base_row["feature_set"],
        "baseline_f1": base_row["f1_tscv_mean"],
        "lstm_f1": lstm_f1,
        "delta_f1": delta_f1,
        "delta_f1_threshold": DELTA_F1_THRESHOLD,
        "delta_f1_pass": (
            bool(delta_f1 >= DELTA_F1_THRESHOLD)
            if not pd.isna(delta_f1)
            else np.nan
        ),
        "lstm_f1_se": lstm_f1_se,
        "delta_f1_se_ratio": ratio,
        "ratio_threshold": DELTA_F1_SE_RATIO_THRESHOLD,
        "ratio_pass": (
            bool(ratio >= DELTA_F1_SE_RATIO_THRESHOLD)
            if not pd.isna(ratio)
            else np.nan
        ),
        "ratio_denominator": "lstm_f1_se",
        "ratio_semantics": (
            "heuristica_descritiva_nao_pareada_"
            "baseada_no_se_da_lstm"
        ),
        "comparison_pairing": (
            "nao_pareada_contra_nb11_full"
        ),
        "ratio_not_interpreted_as": (
            "significancia_estatistica"
        ),
        "governance_decision": decision,
        "recommended_score_source": (
            recommended_score_source
        ),
        "score_source_for_nb14": actual_score_source,
        "score_source_file_for_nb14": actual_score_file,
        "recommended_source_family": (
            recommended_family
        ),
        "actual_source_family": actual_family,
        "score_source_coherence": (
            score_source_coherence
        ),
        "lstm_f1_evidence": (
            f"{lstm_f1_source}:{lstm_f1_field}"
        ),
        "delta_f1_evidence": (
            f"{delta_source}:{delta_field}"
        ),
        "lstm_f1_se_evidence": (
            f"{lstm_f1_se_source}:{lstm_f1_se_field}"
        ),
        "ratio_evidence": (
            f"{ratio_source}:{ratio_field}"
        ),
        "decision_evidence": (
            f"{decision_source}:{decision_field}"
        ),
        "recommendation_evidence": (
            f"{recommended_source_evidence}:"
            f"{recommended_source_field}"
        ),
        "actual_source_evidence": (
            "nb14_scenario_summary:"
            "score_source/is_primary_score_source"
        ),
    })

model_governance = pd.DataFrame(governance_rows)

# ------------------------------------------------------------
# Quatro posições distintas da governança da LSTM
# ------------------------------------------------------------

modelable_cell_ids = sorted(
    nominal_winners["cell_id"].astype(str).tolist()
)

# 1. Passagem apenas pelo primeiro gate nominal ΔF1 >= 0,03.
model_governance["nominal_delta_gate_pass"] = (
    model_governance["governance_decision"]
    == "lstm_scores_can_feed_nb14"
)

nominal_delta_gate_cells = sorted(
    model_governance.loc[
        model_governance["nominal_delta_gate_pass"],
        "cell_id",
    ].tolist()
)

# 2. Passagem pelo gate de robustez descritivo ΔF1/SE >= 2.
model_governance["robustness_gate_pass"] = (
    model_governance["ratio_pass"]
    .fillna(False)
    .astype(bool)
)

robustness_gate_cells = sorted(
    model_governance.loc[
        model_governance["robustness_gate_pass"],
        "cell_id",
    ].tolist()
)

# 3. Promoção efetiva como fonte primária no NB14_FULL.
lstm_source_families = {"NB13_FULL", "NB13A_FULL"}

model_governance["promoted_as_primary_in_nb14"] = (
    model_governance["actual_source_family"]
    .isin(lstm_source_families)
)

promoted_lstm_primary_cells = sorted(
    model_governance.loc[
        model_governance["promoted_as_primary_in_nb14"],
        "cell_id",
    ].tolist()
)

# 4. Avaliação secundária posterior no NB14_FULL.
nb14_lstm_secondary = nb14_summary[
    (~nb14_summary["is_primary_normalized"])
    & (
        nb14_summary["score_source"]
        .map(source_family)
        .isin(lstm_source_families)
    )
].copy()

secondary_lstm_sources_by_cell = (
    nb14_lstm_secondary
    .groupby("cell_id")["score_source"]
    .agg(
        lambda values: "|".join(
            sorted({
                str(value).strip()
                for value in values
                if not pd.isna(value)
            })
        )
    )
    .to_dict()
)

secondary_lstm_evaluation_cells = sorted(
    set(nb14_lstm_secondary["cell_id"].astype(str))
    & set(modelable_cell_ids)
)

model_governance["evaluated_as_secondary_in_nb14"] = (
    model_governance["cell_id"]
    .isin(secondary_lstm_evaluation_cells)
)

model_governance["secondary_lstm_sources_in_nb14"] = (
    model_governance["cell_id"]
    .map(secondary_lstm_sources_by_cell)
    .fillna("")
)

# Posição final inequívoca.
model_governance["final_lstm_position"] = np.select(
    [
        model_governance["promoted_as_primary_in_nb14"],
        model_governance["evaluated_as_secondary_in_nb14"],
    ],
    [
        "promoted_as_primary",
        "not_promoted_secondary_evaluation_only",
    ],
    default="not_promoted_not_evaluated_downstream",
)

lstm_governance_summary = {
    "modelable_cells": modelable_cell_ids,
    "nominal_delta_gate_cells": nominal_delta_gate_cells,
    "robustness_gate_cells": robustness_gate_cells,
    "promoted_lstm_primary_cells": promoted_lstm_primary_cells,
    "secondary_lstm_evaluation_cells": (
        secondary_lstm_evaluation_cells
    ),
    "secondary_sources_by_cell": (
        secondary_lstm_sources_by_cell
    ),
    "final_position": (
        "Nenhuma LSTM foi promovida como fonte primária. "
        f"Células no gate nominal ΔF1 >= 0,03: {nominal_delta_gate_cells}. "
        f"Células no gate de robustez ΔF1/SE >= 2: {robustness_gate_cells}. "
        "NB13_FULL e NB13a_FULL são preservados como análises "
        "secundárias quando não promovidos."
    ),
}

PATH_MODEL_GOVERNANCE = write_csv(
    model_governance,
    "AUD02_model_governance_by_cell.csv",
)

PATH_LSTM_GOVERNANCE_SUMMARY = write_json(
    lstm_governance_summary,
    "AUD02_lstm_governance_summary.json",
)

display(model_governance)
display(pd.DataFrame([lstm_governance_summary]))


# ==============================================================================
# NOTA/SEÇÃO DOCUMENTAL 9
# ==============================================================================
# Módulo 6 — Lift de PR-AUC sobre prevalência
#
# Decisão de protocolo da v1.6:
# - lift oficial da dissertação: teste fixo herdado;
# - numerador: AP rederivada dos escores calibrados congelados;
# - denominador: prevalência das mesmas linhas;
# - diagnóstico complementar: lift TSCV por dobra;
# - legado: razão mista da v1.5 preservada apenas para rastreabilidade.
#
# A escolha pelo teste fixo não afirma que todas as métricas da dissertação
# devam usar o mesmo protocolo. A seleção e a comparação de ROC-AUC do item
# 4.9 permanecem no TSCV; o lift é fixado no teste herdado porque essa opção:
# 1. usa exatamente as mesmas observações no numerador e denominador;
# 2. elimina a ambiguidade razão-das-médias versus média-das-razões;
# 3. conecta o lift à avaliação operacional do NB14_FULL.
#
# Esta rederivação é somente-leitura: não treina, não redefine split,
# não recalibra e não altera qualquer artefato oficial.


# ============================================================
# Módulo 6 — lift alinhado no teste fixo + diagnósticos
# ============================================================

FIXED_TEST_VALIDATION = "fixed_inherited_80_20"
NUMERIC_RTOL = 1e-10
NUMERIC_ATOL = 1e-12

score_columns = [
    "cell_id",
    "scenario_label",
    "feature_set",
    "model",
    "validation",
    "y_true",
    "score_raw",
    "score_calibrated",
    "n_positivos_test",
    "is_modelable",
]

fixed_scores = read_parquet_source(
    "full_scores_calibrated",
    columns=score_columns,
)

require_columns(
    fixed_scores,
    "full_scores_calibrated",
    required=[
        "cell_id", "scenario_label", "feature_set", "model",
        "validation", "y_true", "score_raw", "score_calibrated",
    ],
    optional=["n_positivos_test", "is_modelable"],
)

fixed_scores["cell_id"] = (
    fixed_scores["cell_id"]
    .astype(str)
    .str.strip()
    .str.lower()
)
fixed_scores["model"] = (
    fixed_scores["model"]
    .map(canonical_model_name)
)
fixed_scores["validation"] = (
    fixed_scores["validation"]
    .astype(str)
    .str.strip()
    .str.lower()
)
fixed_scores = to_numeric(
    fixed_scores,
    [
        "y_true",
        "score_raw",
        "score_calibrated",
        "n_positivos_test",
    ],
)

fixed_scores = fixed_scores[
    fixed_scores["validation"]
    == FIXED_TEST_VALIDATION
].copy()

if fixed_scores.empty:
    raise ValueError(
        "Nenhuma linha encontrada para "
        f"validation={FIXED_TEST_VALIDATION}."
    )

# A seleção soberana vem do NB11_FULL e deve coincidir com a fonte
# primária efetivamente consumida pelo NB14_FULL.
primary_by_cell = (
    nb14_primary[
        [
            "cell_id",
            "scenario_label",
            "score_source",
            "is_primary_score_source",
            "score_source_file",
            "model",
            "feature_set",
            "n_scored_rows",
            "positive_rate_tau_reference",
        ]
    ]
    .copy()
)

primary_by_cell["model"] = (
    primary_by_cell["model"]
    .map(canonical_model_name)
)
primary_by_cell["n_scored_rows"] = pd.to_numeric(
    primary_by_cell["n_scored_rows"],
    errors="coerce",
)
primary_by_cell["positive_rate_tau_reference"] = pd.to_numeric(
    primary_by_cell["positive_rate_tau_reference"],
    errors="coerce",
)

official_lift_rows = []

for _, winner in nominal_winners.sort_values("cell_id").iterrows():
    cell_id = str(winner["cell_id"])
    scenario_label = str(winner["scenario_label"])
    feature_set = str(winner["feature_set"])
    model = canonical_model_name(winner["model"])

    primary_match = primary_by_cell[
        primary_by_cell["cell_id"] == cell_id
    ].copy()

    if len(primary_match) != 1:
        raise ValueError(
            "Esperada exatamente uma fonte primária NB14_FULL "
            f"para cell_id={cell_id}; observado={len(primary_match)}."
        )

    primary_row = primary_match.iloc[0]

    selected = fixed_scores[
        (fixed_scores["cell_id"] == cell_id)
        & (fixed_scores["scenario_label"] == scenario_label)
        & (fixed_scores["feature_set"] == feature_set)
        & (fixed_scores["model"] == model)
    ].copy()

    if selected.empty:
        raise ValueError(
            "Escores do resultado soberano não encontrados: "
            f"cell={cell_id}, scenario={scenario_label}, "
            f"feature_set={feature_set}, model={model}."
        )

    y_true = selected["y_true"].to_numpy(dtype=float)
    score_raw = selected["score_raw"].to_numpy(dtype=float)
    score_calibrated = selected[
        "score_calibrated"
    ].to_numpy(dtype=float)

    y_binary = bool(
        set(np.unique(y_true[~np.isnan(y_true)]))
        .issubset({0.0, 1.0})
    )
    scores_finite = bool(
        np.isfinite(score_raw).all()
        and np.isfinite(score_calibrated).all()
    )

    if not y_binary:
        raise ValueError(
            f"y_true não binário na célula {cell_id}."
        )
    if not scores_finite:
        raise ValueError(
            f"Escores não finitos na célula {cell_id}."
        )

    n_rows_observed = int(len(selected))
    n_positive_observed = int(np.sum(y_true))
    prevalence_fixed = float(np.mean(y_true))

    ap_raw = float(
        average_precision_score(y_true, score_raw)
    )
    ap_calibrated = float(
        average_precision_score(
            y_true,
            score_calibrated,
        )
    )

    ap_raw_equals_calibrated = bool(
        np.isclose(
            ap_raw,
            ap_calibrated,
            rtol=NUMERIC_RTOL,
            atol=NUMERIC_ATOL,
        )
    )

    lift_fixed = (
        ap_calibrated / prevalence_fixed
        if prevalence_fixed > 0
        else np.nan
    )

    n_rows_nb14 = int(
        primary_row["n_scored_rows"]
    )
    prevalence_nb14 = float(
        primary_row[
            "positive_rate_tau_reference"
        ]
    )

    n_positive_nb11 = pd.to_numeric(
        pd.Series([
            winner.get("n_positivos_test", np.nan)
        ]),
        errors="coerce",
    ).iloc[0]

    parquet_positive_metadata = (
        selected["n_positivos_test"]
        .dropna()
        .unique()
        .tolist()
        if "n_positivos_test" in selected.columns
        else []
    )

    if len(parquet_positive_metadata) == 1:
        n_positive_parquet_metadata = int(
            parquet_positive_metadata[0]
        )
    else:
        n_positive_parquet_metadata = np.nan

    primary_source_family = source_family(
        primary_row["score_source"]
    )

    primary_scenario_matches_winner = (
        str(primary_row["scenario_label"])
        == scenario_label
    )
    primary_model_matches_winner = (
        canonical_model_name(
            primary_row["model"]
        )
        == model
    )
    primary_feature_matches_winner = (
        str(primary_row["feature_set"])
        == feature_set
    )

    row_count_matches_nb14 = (
        n_rows_observed == n_rows_nb14
    )
    prevalence_matches_nb14 = bool(
        np.isclose(
            prevalence_fixed,
            prevalence_nb14,
            rtol=NUMERIC_RTOL,
            atol=NUMERIC_ATOL,
        )
    )
    positive_count_matches_nb11 = (
        bool(
            n_positive_observed
            == int(n_positive_nb11)
        )
        if not pd.isna(n_positive_nb11)
        else False
    )
    positive_count_matches_parquet_metadata = (
        bool(
            n_positive_observed
            == n_positive_parquet_metadata
        )
        if not pd.isna(
            n_positive_parquet_metadata
        )
        else False
    )
    primary_source_is_nb11 = (
        primary_source_family == "NB11_FULL"
    )
    primary_flag_is_true = bool(
        normalize_bool(
            pd.Series([
                primary_row[
                    "is_primary_score_source"
                ]
            ])
        ).iloc[0]
    )

    official_lift_rows.append({
        "cell_id": cell_id,
        "scenario_label": scenario_label,
        "feature_set": feature_set,
        "model": model,
        "validation": FIXED_TEST_VALIDATION,
        "score_source": primary_row["score_source"],
        "score_source_family": primary_source_family,
        "score_source_file_nb14": (
            primary_row["score_source_file"]
        ),
        "score_artifact_aggregate": str(
            SOURCE_REGISTRY[
                "full_scores_calibrated"
            ]
        ),
        "n_scored_rows_observed": n_rows_observed,
        "n_scored_rows_nb14": n_rows_nb14,
        "row_count_matches_nb14": (
            row_count_matches_nb14
        ),
        "n_positive_test_observed": (
            n_positive_observed
        ),
        "n_positive_test_nb11": (
            None
            if pd.isna(n_positive_nb11)
            else int(n_positive_nb11)
        ),
        "n_positive_test_parquet_metadata": (
            None
            if pd.isna(
                n_positive_parquet_metadata
            )
            else int(
                n_positive_parquet_metadata
            )
        ),
        "positive_count_matches_nb11": (
            positive_count_matches_nb11
        ),
        "positive_count_matches_parquet_metadata": (
            positive_count_matches_parquet_metadata
        ),
        "prevalence_fixed_test": prevalence_fixed,
        "prevalence_nb14": prevalence_nb14,
        "prevalence_matches_nb14": (
            prevalence_matches_nb14
        ),
        "average_precision_score_raw": ap_raw,
        "average_precision_score_calibrated": (
            ap_calibrated
        ),
        "ap_raw_equals_calibrated": (
            ap_raw_equals_calibrated
        ),
        "lift_pr_fixed_test": lift_fixed,
        "formula": (
            "average_precision_score("
            "y_true, score_calibrated"
            ") / mean(y_true)"
        ),
        "protocol_alignment": (
            "ALIGNED_NB11_FIXED_INHERITED_TEST"
        ),
        "primary_source_is_nb11": (
            primary_source_is_nb11
        ),
        "primary_flag_is_true": (
            primary_flag_is_true
        ),
        "primary_scenario_matches_winner": (
            primary_scenario_matches_winner
        ),
        "primary_model_matches_winner": (
            primary_model_matches_winner
        ),
        "primary_feature_matches_winner": (
            primary_feature_matches_winner
        ),
        "y_true_binary": y_binary,
        "scores_finite": scores_finite,
        "is_modelable": True,
    })

pr_auc_lift_fixed_test = pd.DataFrame(
    official_lift_rows
).sort_values("cell_id").reset_index(drop=True)

# Alias oficial usado pelas claims e pelo resumo.
pr_auc_lift = pr_auc_lift_fixed_test.copy()
pr_auc_lift["pr_auc_value"] = (
    pr_auc_lift[
        "average_precision_score_calibrated"
    ]
)
pr_auc_lift["prevalence_value"] = (
    pr_auc_lift["prevalence_fixed_test"]
)
pr_auc_lift["lift_pr"] = (
    pr_auc_lift["lift_pr_fixed_test"]
)
pr_auc_lift["pr_auc_field"] = (
    "derived:average_precision_score("
    "y_true,score_calibrated)"
)
pr_auc_lift["pr_auc_file"] = str(
    SOURCE_REGISTRY[
        "full_scores_calibrated"
    ]
)
pr_auc_lift["prevalence_field"] = (
    "derived:mean(y_true)"
)
pr_auc_lift["prevalence_file"] = str(
    SOURCE_REGISTRY[
        "full_scores_calibrated"
    ]
)

PATH_PR_AUC_LIFT_FIXED = write_csv(
    pr_auc_lift_fixed_test,
    "AUD02_pr_auc_lift_fixed_test_by_cell.csv",
)
PATH_PR_AUC_LIFT = write_csv(
    pr_auc_lift,
    "AUD02_pr_auc_lift_by_cell.csv",
)

# ------------------------------------------------------------
# Legado v1.5 — protocolo misto preservado para rastreabilidade
# ------------------------------------------------------------

legacy_rows = []

for _, winner in nominal_winners.sort_values(
    "cell_id"
).iterrows():
    cell_id = str(winner["cell_id"])
    primary_row = primary_by_cell[
        primary_by_cell["cell_id"] == cell_id
    ].iloc[0]

    ap_tscv_mean = float(
        winner["average_precision_tscv_mean"]
    )
    prevalence_nb14 = float(
        primary_row[
            "positive_rate_tau_reference"
        ]
    )
    mixed_lift = (
        ap_tscv_mean / prevalence_nb14
        if prevalence_nb14 > 0
        else np.nan
    )

    fixed_row = pr_auc_lift_fixed_test[
        pr_auc_lift_fixed_test["cell_id"]
        == cell_id
    ].iloc[0]

    fixed_lift = float(
        fixed_row["lift_pr_fixed_test"]
    )

    legacy_rows.append({
        "cell_id": cell_id,
        "scenario_label": winner["scenario_label"],
        "feature_set": winner["feature_set"],
        "model": winner["model"],
        "average_precision_tscv_mean": (
            ap_tscv_mean
        ),
        "prevalence_fixed_test_nb14": (
            prevalence_nb14
        ),
        "lift_mixed_legacy": mixed_lift,
        "lift_fixed_test_official": fixed_lift,
        "absolute_difference_fixed_minus_mixed": (
            fixed_lift - mixed_lift
        ),
        "relative_difference_pct": (
            100.0
            * (fixed_lift - mixed_lift)
            / mixed_lift
            if mixed_lift != 0
            else np.nan
        ),
        "legacy_protocol": (
            "MIXED_TSCV_AP_VS_NB14_"
            "FIXED_TEST_PREVALENCE"
        ),
        "official_protocol": (
            "ALIGNED_NB11_FIXED_INHERITED_TEST"
        ),
        "official_use": False,
    })

pr_auc_lift_mixed_legacy = pd.DataFrame(
    legacy_rows
)

PATH_PR_AUC_LIFT_LEGACY = write_csv(
    pr_auc_lift_mixed_legacy,
    "AUD02_pr_auc_lift_mixed_legacy_by_cell.csv",
)

# ------------------------------------------------------------
# Opção A — diagnóstico TSCV, não métrica oficial
# ------------------------------------------------------------

require_columns(
    raw_tscv,
    "full_metrics_tscv",
    required=[
        "cell_id", "scenario_label", "feature_set",
        "model", "validation", "fold",
        "average_precision",
    ],
    optional=[
        "test_positive_rate",
        "validation_positive_rate",
        "positive_rate",
        "n_test",
    ],
)

tscv_prevalence_field = first_existing(
    raw_tscv.columns,
    [
        "test_positive_rate",
        "validation_positive_rate",
        "positive_rate",
    ],
)

if tscv_prevalence_field is None:
    raise KeyError(
        "Arquivo TSCV sem taxa de positivos "
        "da dobra de validação."
    )

raw_tscv["validation"] = (
    raw_tscv["validation"]
    .astype(str)
    .str.strip()
    .str.lower()
)
raw_tscv = to_numeric(
    raw_tscv,
    [
        "fold",
        "average_precision",
        tscv_prevalence_field,
        "n_test",
    ],
)

tscv_diag_rows = []

for _, winner in nominal_winners.sort_values(
    "cell_id"
).iterrows():
    cell_id = str(winner["cell_id"])
    model = canonical_model_name(
        winner["model"]
    )

    folds = raw_tscv[
        (raw_tscv["cell_id"] == cell_id)
        & (
            raw_tscv["scenario_label"]
            == winner["scenario_label"]
        )
        & (
            raw_tscv["feature_set"]
            == winner["feature_set"]
        )
        & (raw_tscv["model"] == model)
        & (
            raw_tscv["validation"]
            == "timeseries_split"
        )
    ].copy()

    folds = folds[
        folds["average_precision"].notna()
        & folds[
            tscv_prevalence_field
        ].notna()
        & (
            folds[
                tscv_prevalence_field
            ] > 0
        )
    ].copy()

    if folds.empty:
        raise ValueError(
            "Dobras TSCV válidas não encontradas "
            f"para cell_id={cell_id}."
        )

    folds["lift_pr_fold"] = (
        folds["average_precision"]
        / folds[tscv_prevalence_field]
    )

    ap_mean = float(
        folds["average_precision"].mean()
    )
    prevalence_mean = float(
        folds[
            tscv_prevalence_field
        ].mean()
    )
    lift_ratio_of_means = (
        ap_mean / prevalence_mean
    )
    lift_mean_of_ratios = float(
        folds["lift_pr_fold"].mean()
    )

    ap_summary = float(
        winner[
            "average_precision_tscv_mean"
        ]
    )

    tscv_diag_rows.append({
        "cell_id": cell_id,
        "scenario_label": winner["scenario_label"],
        "feature_set": winner["feature_set"],
        "model": model,
        "validation": "timeseries_split",
        "prevalence_field": (
            tscv_prevalence_field
        ),
        "n_folds": int(
            folds["fold"].nunique()
        ),
        "average_precision_fold_mean": (
            ap_mean
        ),
        "average_precision_summary": (
            ap_summary
        ),
        "ap_fold_mean_matches_summary": bool(
            np.isclose(
                ap_mean,
                ap_summary,
                rtol=NUMERIC_RTOL,
                atol=NUMERIC_ATOL,
            )
        ),
        "prevalence_fold_mean": (
            prevalence_mean
        ),
        "lift_ratio_of_means": (
            lift_ratio_of_means
        ),
        "lift_mean_of_fold_ratios": (
            lift_mean_of_ratios
        ),
        "lift_fold_median": float(
            folds["lift_pr_fold"].median()
        ),
        "lift_fold_std": float(
            folds["lift_pr_fold"].std(
                ddof=1
            )
        ),
        "lift_fold_min": float(
            folds["lift_pr_fold"].min()
        ),
        "lift_fold_max": float(
            folds["lift_pr_fold"].max()
        ),
        "aggregation_ambiguity_absolute": (
            lift_mean_of_ratios
            - lift_ratio_of_means
        ),
        "role": (
            "diagnostico_complementar_nao_oficial"
        ),
        "protocol_alignment": (
            "ALIGNED_TSCV_FOLD_VALIDATION"
        ),
    })

pr_auc_lift_tscv_diagnostic = pd.DataFrame(
    tscv_diag_rows
)

PATH_PR_AUC_LIFT_TSCV = write_csv(
    pr_auc_lift_tscv_diagnostic,
    "AUD02_pr_auc_lift_tscv_diagnostic_by_cell.csv",
)

lift_protocol_decision = {
    "official_protocol": (
        "ALIGNED_NB11_FIXED_INHERITED_TEST"
    ),
    "official_formula": (
        "average_precision_score("
        "y_true, score_calibrated"
        ") / mean(y_true)"
    ),
    "official_reason": [
        (
            "numerador e denominador usam "
            "exatamente as mesmas observações"
        ),
        (
            "não há ambiguidade entre razão "
            "das médias e média das razões"
        ),
        (
            "alinhamento com a avaliação "
            "operacional do NB14_FULL"
        ),
    ],
    "tscv_diagnostic_role": (
        "diagnostico complementar de "
        "discriminacao/ordenacao"
    ),
    "cross_metric_protocol_policy": {
        "model_selection_and_item_4_9_roc_auc": (
            "TSCV"
        ),
        "official_pr_auc_lift": (
            "fixed_inherited_80_20"
        ),
        "operational_metrics_nb14": (
            "fixed_inherited_80_20"
        ),
        "interpretation": (
            "protocolos diferentes por finalidade "
            "da métrica; nunca misturados dentro "
            "do mesmo cálculo"
        ),
    },
    "legacy_mixed_protocol": (
        "preservado apenas para rastreabilidade; "
        "não utilizar como número oficial"
    ),
    "read_only_assurance": (
        "nenhum treino, split, calibração, "
        "seleção ou artefato oficial foi alterado"
    ),
}

PATH_LIFT_PROTOCOL_DECISION = write_json(
    lift_protocol_decision,
    "AUD02_lift_protocol_decision.json",
)

lift_min = float(
    pr_auc_lift["lift_pr"].min()
)
lift_max = float(
    pr_auc_lift["lift_pr"].max()
)
legacy_lift_min = float(
    pr_auc_lift_mixed_legacy[
        "lift_mixed_legacy"
    ].min()
)
legacy_lift_max = float(
    pr_auc_lift_mixed_legacy[
        "lift_mixed_legacy"
    ].max()
)

ap_raw_calibrated_equal_all = bool(
    pr_auc_lift_fixed_test[
        "ap_raw_equals_calibrated"
    ].all()
)

official_lift_row_checks = [
    "row_count_matches_nb14",
    "positive_count_matches_nb11",
    "positive_count_matches_parquet_metadata",
    "prevalence_matches_nb14",
    "primary_source_is_nb11",
    "primary_flag_is_true",
    "primary_scenario_matches_winner",
    "primary_model_matches_winner",
    "primary_feature_matches_winner",
    "y_true_binary",
    "scores_finite",
]

official_lift_valid = bool(
    len(pr_auc_lift_fixed_test)
    == len(modelable_cell_ids)
    and (
        pr_auc_lift_fixed_test[
            "protocol_alignment"
        ]
        == (
            "ALIGNED_NB11_FIXED_"
            "INHERITED_TEST"
        )
    ).all()
    and pr_auc_lift_fixed_test[
        "lift_pr_fixed_test"
    ].notna().all()
    and all(
        pr_auc_lift_fixed_test[
            column
        ].all()
        for column in official_lift_row_checks
    )
)

display(
    pr_auc_lift_fixed_test.style.format({
        "prevalence_fixed_test": "{:.6f}",
        "average_precision_score_raw": "{:.6f}",
        "average_precision_score_calibrated": "{:.6f}",
        "lift_pr_fixed_test": "{:.3f}",
    })
)

display(
    pr_auc_lift_tscv_diagnostic.style.format({
        "average_precision_fold_mean": "{:.6f}",
        "prevalence_fold_mean": "{:.6f}",
        "lift_ratio_of_means": "{:.3f}",
        "lift_mean_of_fold_ratios": "{:.3f}",
    })
)

print(
    "Lift oficial no teste fixo:",
    f"{lift_min:.6f} a {lift_max:.6f}",
)
print(
    "Faixa legada mista:",
    f"{legacy_lift_min:.6f} a "
    f"{legacy_lift_max:.6f}",
)
print(
    "AP(raw) == AP(calibrated) em todas as células:",
    ap_raw_calibrated_equal_all,
)


# ==============================================================================
# NOTA/SEÇÃO DOCUMENTAL 10
# ==============================================================================
# Módulo 8 — Métricas operacionais e antecipação
# A taxa de antecipação é registrada com:
# - limiar \(\tau\);
# - episódios avaliáveis;
# - episódios antecipados;
# - razão reproduzida;
# - regra para episódios não avaliáveis.
# A mediana é calculada entre as células modeláveis na fonte primária do NB14.


# ============================================================
# Módulo 8 — antecipação e métricas operacionais
# ============================================================

operational_columns = [
    "cell_id", "scenario_label", "score_source",
    "score_source_file", "model", "feature_set",
    "n_scored_rows", "n_total_episodes",
    "duration_one_window_pct", "tau_reference",
    "precision_tau_reference", "recall_tau_reference",
    "f1_tau_reference",
    "false_alerts_per_day_tau_reference",
    "positive_rate_tau_reference",
    "alert_rate_tau_reference",
    "n_evaluable_episodes_tau_reference",
    "n_anticipated_episodes_tau_reference",
    "episode_anticipation_rate_tau_reference",
    "lead_time_minutes_mean_tau_reference",
    "lead_time_minutes_median_tau_reference",
]
operational_columns = [
    c for c in operational_columns
    if c in nb14_primary.columns
]

operational_metrics = (
    nb14_primary[operational_columns]
    .merge(
        modelability_summary[
            ["cell_id", "is_modelable"]
        ],
        on="cell_id",
        how="left",
        validate="one_to_one",
    )
    .sort_values("cell_id")
    .reset_index(drop=True)
)

operational_metrics = to_numeric(
    operational_metrics,
    [
        "tau_reference",
        "n_evaluable_episodes_tau_reference",
        "n_anticipated_episodes_tau_reference",
        "episode_anticipation_rate_tau_reference",
    ],
)

operational_metrics[
    "anticipation_rate_recomputed"
] = np.where(
    operational_metrics[
        "n_evaluable_episodes_tau_reference"
    ] > 0,
    (
        operational_metrics[
            "n_anticipated_episodes_tau_reference"
        ]
        / operational_metrics[
            "n_evaluable_episodes_tau_reference"
        ]
    ),
    np.nan,
)

operational_metrics[
    "anticipation_rate_matches"
] = np.isclose(
    operational_metrics[
        "anticipation_rate_recomputed"
    ],
    operational_metrics[
        "episode_anticipation_rate_tau_reference"
    ],
    rtol=1e-10,
    atol=1e-12,
    equal_nan=True,
)

modelable_operational = operational_metrics[
    normalize_bool(
        operational_metrics["is_modelable"]
    )
].copy()

anticipation_median = float(
    modelable_operational[
        "episode_anticipation_rate_tau_reference"
    ].median()
)

anticipation_claim_audit = pd.DataFrame([{
    "claim_id": "4.4",
    "statistic": "median_between_modelable_cells",
    "tau_semantics": "tau_reference_per_primary_NB14_row",
    "n_cells": int(
        modelable_operational["cell_id"].nunique()
    ),
    "median_episode_anticipation_rate": (
        anticipation_median
    ),
    "median_percentage": 100.0 * anticipation_median,
    "numerator_field": (
        "n_anticipated_episodes_tau_reference"
    ),
    "denominator_field": (
        "n_evaluable_episodes_tau_reference"
    ),
    "rate_field": (
        "episode_anticipation_rate_tau_reference"
    ),
    "source_file": str(
        SOURCE_REGISTRY["nb14_scenario_summary"]
    ),
    "all_cell_rates_recomputed": bool(
        modelable_operational[
            "anticipation_rate_matches"
        ].all()
    ),
    "non_evaluable_rule": (
        "rate_is_nan_when_n_evaluable_episodes_is_zero"
    ),
}])

PATH_OPERATIONAL = write_csv(
    operational_metrics,
    "AUD02_operational_metrics_by_cell.csv",
)
PATH_ANTICIPATION = write_csv(
    anticipation_claim_audit,
    "AUD02_anticipation_claim_audit.csv",
)

display(operational_metrics)
display(anticipation_claim_audit)


# ==============================================================================
# NOTA/SEÇÃO DOCUMENTAL 11
# ==============================================================================
# Módulo 1 — Inventário, integridade e manifestos a montante
# O AUD02 é posterior ao fechamento experimental, porém não consome o NB16_FULL.
# A coerência com manifestos significa:
# - localizar o manifesto da etapa a montante, quando disponível;
# - procurar o arquivo-fonte pelo nome;
# - comparar o SHA-256 registrado com o atual;
# - registrar NOT_LISTED ou UNRESOLVED_SCHEMA sem inventar conformidade.


# ============================================================
# Módulo 1 — inventário, hashes, schemas e manifestos
# ============================================================

def infer_stage_root(path: Path) -> Path | None:
    resolved = path.resolve()
    for parent in [resolved.parent, *resolved.parents]:
        if re.match(
            r"^(0[4-9]|1[0-5])[a-z]?_full_",
            parent.name.lower(),
        ):
            return parent
    return None

def find_stage_manifest(path: Path) -> Path | None:
    stage_root = infer_stage_root(path)
    if stage_root is None:
        return None

    candidates = sorted(
        p for p in stage_root.rglob(
            "*artifact_manifest_sha256.csv"
        )
        if p.is_file() and valid_search_candidate(p)
    )
    return candidates[0].resolve() if candidates else None

def check_manifest_entry(
    source_path: Path,
) -> dict[str, Any]:
    manifest_path = find_stage_manifest(source_path)

    if manifest_path is None:
        return {
            "manifest_path": None,
            "manifest_status": "NO_MANIFEST_FOUND",
            "manifest_sha256": None,
        }

    try:
        manifest = pd.read_csv(
            manifest_path,
            encoding="utf-8-sig",
            low_memory=False,
        )
    except Exception as exc:
        return {
            "manifest_path": str(manifest_path),
            "manifest_status": (
                f"READ_ERROR:{type(exc).__name__}"
            ),
            "manifest_sha256": None,
        }

    file_col = first_existing(
        manifest.columns,
        [
            "file", "filename", "file_name", "artifact",
            "artifact_path", "relative_path", "path",
        ],
    )
    hash_col = first_existing(
        manifest.columns,
        [
            "sha256", "sha_256",
            "hash_sha256", "file_sha256",
        ],
    )

    if file_col is None or hash_col is None:
        return {
            "manifest_path": str(manifest_path),
            "manifest_status": "UNRESOLVED_SCHEMA",
            "manifest_sha256": None,
        }

    names = manifest[file_col].astype(str)
    rows = manifest[
        names.eq(source_path.name)
        | names.str.endswith("/" + source_path.name)
        | names.str.endswith(
            "\\" + source_path.name
        )
    ]

    if rows.empty:
        return {
            "manifest_path": str(manifest_path),
            "manifest_status": "NOT_LISTED",
            "manifest_sha256": None,
        }

    recorded = str(
        rows.iloc[0][hash_col]
    ).strip().lower()
    current = sha256_file(source_path).lower()

    return {
        "manifest_path": str(manifest_path),
        "manifest_status": (
            "MATCH" if recorded == current
            else "MISMATCH"
        ),
        "manifest_sha256": recorded,
    }

inventory_rows = []
hash_rows = []

for key, path in sorted(SOURCE_REGISTRY.items()):
    fp = fingerprint(path)
    manifest_info = check_manifest_entry(path)

    inventory_rows.append({
        "source_key": key,
        "stage": next(
            (
                spec.stage for spec in SPECS
                if spec.key == key
            ),
            "",
        ),
        "description": next(
            (
                spec.description for spec in SPECS
                if spec.key == key
            ),
            "",
        ),
        "path": str(path),
        "filename": path.name,
        "size_bytes": fp["size_bytes"],
        "modified_at": datetime.fromtimestamp(
            path.stat().st_mtime,
            tz=timezone.utc,
        ).astimezone().isoformat(),
        **manifest_info,
    })

    hash_rows.append({
        "source_key": key,
        "path": str(path),
        "sha256": fp["sha256"],
    })

source_inventory = pd.DataFrame(inventory_rows)
source_hashes = pd.DataFrame(hash_rows)
schema_checks = pd.DataFrame(SCHEMA_RESULTS)

PATH_SOURCE_INVENTORY = write_csv(
    source_inventory,
    "AUD02_source_inventory.csv",
)
PATH_SOURCE_HASHES = write_csv(
    source_hashes,
    "AUD02_source_hashes.csv",
)
PATH_SCHEMA_CHECKS = write_csv(
    schema_checks,
    "AUD02_schema_checks.csv",
)

display(source_inventory)
display(schema_checks)


# ==============================================================================
# NOTA/SEÇÃO DOCUMENTAL 12
# ==============================================================================
# Checks e snapshot de regressão
# Classes
# - STRUCTURAL — falha indica violação da arquitetura ou integridade.
# - REGRESSION — divergência indica mudança a investigar; gera REVIEW.
# - TRACEABILITY — verifica arquivo, campo, regra e semântica.
# Política do snapshot
# - criado somente se não existir;
# - nunca sobrescrito automaticamente;
# - atualização exige justificativa datada e novo versionamento;
# - o notebook não “aceita” novos valores por flag.



# ============================================================
# Auditoria do contrato de atributos da linhagem corrigida
# ============================================================
# A verificação é independente do NB16_FULL:
# - semântica n_failed/event_FAIL_count: Parquet model-facing FULL;
# - feature sets vigentes: 11_FULL_feature_sets.json;
# - duplicidade por valores: matriz efetivamente entregue aos modelos no NB11_FULL.

RAW_CORE_EXPECTED = [
    "fail_rate",
    "n_events",
    "n_failed",
    "n_machines",
    "n_collections",
    "event_LOST_count",
]
TEMPORAL_CORE_EXPECTED = RAW_CORE_EXPECTED + [
    "lag_1",
    "lag_2",
    "lag_3",
    "rolling_mean_1h",
    "rolling_std_1h",
    "pct_change",
    "zscore_expanding",
]

def _normalize_feature_set_name(value: Any) -> str:
    return str(value).strip().lower().replace("-", "_").replace(" ", "_")

def _feature_list_from_value(value: Any) -> list[str] | None:
    if isinstance(value, list):
        return [str(v) for v in value]
    if isinstance(value, tuple):
        return [str(v) for v in value]
    if isinstance(value, dict):
        for candidate in (
            "features", "columns", "feature_names", "predictors", "variables"
        ):
            if candidate in value and isinstance(value[candidate], (list, tuple)):
                return [str(v) for v in value[candidate]]
    return None

def _collect_named_feature_sets(
    value: Any,
    target_name: str,
    path: str = "$",
) -> list[dict[str, Any]]:
    target = _normalize_feature_set_name(target_name)
    found: list[dict[str, Any]] = []

    if isinstance(value, dict):
        for key, child in value.items():
            child_path = f"{path}.{key}"
            if _normalize_feature_set_name(key) == target:
                features = _feature_list_from_value(child)
                if features is not None:
                    found.append({
                        "path": child_path,
                        "features": features,
                    })
            found.extend(
                _collect_named_feature_sets(
                    child,
                    target_name=target_name,
                    path=child_path,
                )
            )
    elif isinstance(value, list):
        for idx, child in enumerate(value):
            found.extend(
                _collect_named_feature_sets(
                    child,
                    target_name=target_name,
                    path=f"{path}[{idx}]",
                )
            )
    return found

def _collect_mapping_keys(value: Any) -> list[str]:
    keys: list[str] = []
    if isinstance(value, dict):
        for key, child in value.items():
            keys.append(str(key))
            keys.extend(_collect_mapping_keys(child))
    elif isinstance(value, list):
        for child in value:
            keys.extend(_collect_mapping_keys(child))
    return keys

feature_sets_payload = read_json_source("full_feature_sets")

raw_definitions = _collect_named_feature_sets(
    feature_sets_payload, "raw_core"
)
temporal_definitions = _collect_named_feature_sets(
    feature_sets_payload, "temporal_core"
)

if not raw_definitions:
    raise AssertionError(
        "11_FULL_feature_sets.json não contém definição resolvível de raw_core."
    )
if not temporal_definitions:
    raise AssertionError(
        "11_FULL_feature_sets.json não contém definição resolvível de temporal_core."
    )

def _canonical_definition_records(
    definitions: list[dict[str, Any]],
) -> list[tuple[str, ...]]:
    return sorted({
        tuple(record["features"])
        for record in definitions
    })

raw_unique_definitions = _canonical_definition_records(raw_definitions)
temporal_unique_definitions = _canonical_definition_records(
    temporal_definitions
)

raw_contract_ok = (
    raw_unique_definitions == [tuple(RAW_CORE_EXPECTED)]
)
temporal_contract_ok = (
    temporal_unique_definitions == [tuple(TEMPORAL_CORE_EXPECTED)]
)

all_feature_set_keys = {
    _normalize_feature_set_name(key)
    for key in _collect_mapping_keys(feature_sets_payload)
}
raw_7_active_evidence = "raw_7" in all_feature_set_keys

active_lists = (
    [record["features"] for record in raw_definitions]
    + [record["features"] for record in temporal_definitions]
)
event_fail_predictor_evidence = any(
    "event_FAIL_count" in features
    for features in active_lists
)

# Verificação semântica diretamente no artefato model-facing FULL.
provenance_df = read_parquet_source(
    "full_model_facing",
    columns=["n_failed", "event_FAIL_count"],
)
if len(provenance_df) == 0:
    raise AssertionError("Parquet model-facing FULL está vazio.")

prov_left = pd.to_numeric(
    provenance_df["n_failed"], errors="coerce"
)
prov_right = pd.to_numeric(
    provenance_df["event_FAIL_count"], errors="coerce"
)
prov_equal_mask = (
    prov_left.eq(prov_right)
    | (prov_left.isna() & prov_right.isna())
)
provenance_mismatch_count = int((~prov_equal_mask).sum())
semantic_duplicate_verified = provenance_mismatch_count == 0

# Verificação do vetor efetivamente entregue ao NB11_FULL.
model_input_df = read_parquet_source("full_model_input_features")
missing_temporal_model_input = sorted(
    set(TEMPORAL_CORE_EXPECTED) - set(model_input_df.columns)
)
event_fail_in_model_input = (
    "event_FAIL_count" in model_input_df.columns
)

active_exact_duplicate_pairs: list[tuple[str, str]] = []
if not missing_temporal_model_input:
    numeric_active = model_input_df[
        TEMPORAL_CORE_EXPECTED
    ].apply(pd.to_numeric, errors="coerce")

    for i, left_name in enumerate(TEMPORAL_CORE_EXPECTED):
        left_values = numeric_active[left_name]
        for right_name in TEMPORAL_CORE_EXPECTED[i + 1:]:
            right_values = numeric_active[right_name]
            equal_mask = (
                left_values.eq(right_values)
                | (left_values.isna() & right_values.isna())
            )
            if bool(equal_mask.all()):
                active_exact_duplicate_pairs.append(
                    (left_name, right_name)
                )

active_duplicate_check_ok = (
    not missing_temporal_model_input
    and not event_fail_in_model_input
    and len(active_exact_duplicate_pairs) == 0
)

feature_contract_audit = {
    "source_policy": {
        "max_full_stage": 15,
        "nb16_consumed": False,
        "feature_set_source": str(
            SOURCE_REGISTRY["full_feature_sets"]
        ),
        "model_input_source": str(
            SOURCE_REGISTRY["full_model_input_features"]
        ),
        "provenance_source": str(
            SOURCE_REGISTRY["full_model_facing"]
        ),
    },
    "raw_core_expected": RAW_CORE_EXPECTED,
    "temporal_core_expected": TEMPORAL_CORE_EXPECTED,
    "raw_core_count": len(RAW_CORE_EXPECTED),
    "temporal_core_count": len(TEMPORAL_CORE_EXPECTED),
    "raw_definitions_found": raw_definitions,
    "temporal_definitions_found": temporal_definitions,
    "raw_unique_definitions": [
        list(v) for v in raw_unique_definitions
    ],
    "temporal_unique_definitions": [
        list(v) for v in temporal_unique_definitions
    ],
    "raw_contract_ok": raw_contract_ok,
    "temporal_contract_ok": temporal_contract_ok,
    "raw_7_active_evidence": raw_7_active_evidence,
    "event_FAIL_count_predictor_evidence": (
        event_fail_predictor_evidence
    ),
    "event_FAIL_count_in_model_input": event_fail_in_model_input,
    "provenance_rows_checked": int(len(provenance_df)),
    "provenance_mismatch_count": provenance_mismatch_count,
    "semantic_duplicate_verified": semantic_duplicate_verified,
    "missing_temporal_model_input": missing_temporal_model_input,
    "active_exact_duplicate_pairs": [
        list(pair) for pair in active_exact_duplicate_pairs
    ],
    "active_duplicate_check_ok": active_duplicate_check_ok,
}

write_json(
    feature_contract_audit,
    "AUD02_feature_contract_audit.json",
)


# ============================================================
# Snapshot de regressão versionado
# ============================================================

DEFAULT_EXPECTED_VALUES = {
    "snapshot_id": "AUD02-v1.2-corrected-lineage",
    "created_at": datetime.now(timezone.utc).date().isoformat(),
    "basis": (
        "Primeira auditoria formal da linhagem corrigida, após estabilização do pipeline. "
        "Os números governantes são derivados dos artefatos regenerados."
    ),
    "update_policy": (
        "Não sobrescrever automaticamente após a primeira criação. "
        "Mudanças posteriores exigem novo snapshot e justificativa."
    ),
    "values": {
        "n_modelable_cells": int(len(nominal_winners)),
        "model_frequency": {
            str(k): int(v)
            for k, v in
            nominal_winners["model"].value_counts().to_dict().items()
        },
        "roc_auc_min": float(
            kaggle_comparison.loc[
                kaggle_comparison["is_modelable"].astype(bool),
                "roc_auc_tscv_mean",
            ].min()
        ),
        "roc_auc_max": float(
            kaggle_comparison.loc[
                kaggle_comparison["is_modelable"].astype(bool),
                "roc_auc_tscv_mean",
            ].max()
        ),
        "reference_roc_auc": float(KAGGLE_REFERENCE_ROC_AUC),
        "n_above_reference": int(
            kaggle_comparison["supera_referencia"].fillna(False).sum()
        ),
        "not_above_reference_cells": sorted(
            kaggle_comparison.loc[
                kaggle_comparison["is_modelable"].astype(bool)
                & ~kaggle_comparison["supera_referencia"].fillna(False),
                "cell_id",
            ].astype(str).tolist()
        ),
        "excluded_from_reference_comparison": sorted(
            kaggle_comparison.loc[
                ~kaggle_comparison["is_modelable"].astype(bool),
                "cell_id",
            ].astype(str).tolist()
        ),
    },
}

if EXPECTED_VALUES_PATH.exists():
    with EXPECTED_VALUES_PATH.open(
        "r", encoding="utf-8"
    ) as file_obj:
        expected_snapshot = json.load(file_obj)
    SNAPSHOT_STATUS = "LOADED_EXISTING"
else:
    expected_snapshot = DEFAULT_EXPECTED_VALUES
    write_json(
        expected_snapshot,
        "AUD02_expected_values.json",
    )
    SNAPSHOT_STATUS = "CREATED_INITIAL"

print("SNAPSHOT_STATUS:", SNAPSHOT_STATUS)
display(pd.json_normalize(expected_snapshot))


# ============================================================
# Checks em três classes
# ============================================================

checks: list[dict[str, Any]] = []

def add_check(
    check_id: str,
    check_class: str,
    description: str,
    observed: Any,
    expected: Any,
    status: str,
    evidence: str,
    action: str,
) -> None:
    checks.append({
        "check_id": check_id,
        "check_class": check_class,
        "description": description,
        "observed": observed,
        "expected": expected,
        "status": status,
        "evidence": evidence,
        "action": action,
    })


add_check(
    "S00_FEATURE_CONTRACT",
    "STRUCTURAL",
    (
        "Contrato vigente de atributos confirmado em fontes até NB15_FULL: "
        "raw_core=6, temporal_core=13, sem raw_7 ativo e sem "
        "event_FAIL_count nos preditores."
    ),
    feature_contract_audit,
    {
        "raw_core": RAW_CORE_EXPECTED,
        "temporal_core": TEMPORAL_CORE_EXPECTED,
        "raw_7_active_evidence": False,
        "event_FAIL_count_predictor_evidence": False,
        "event_FAIL_count_in_model_input": False,
        "active_exact_duplicate_pairs": [],
    },
    (
        "PASS"
        if (
            raw_contract_ok
            and temporal_contract_ok
            and not raw_7_active_evidence
            and not event_fail_predictor_evidence
            and active_duplicate_check_ok
        )
        else "FAIL"
    ),
    (
        "11_FULL_feature_sets.json | "
        "11_FULL_model_input_features_by_cell.parquet | "
        "AUD02_feature_contract_audit.json"
    ),
    (
        "Interromper a auditoria e revisar NB11_FULL se o contrato 6/13, "
        "a exclusão de event_FAIL_count ou a ausência de duplicidades "
        "exatas no vetor ativo não forem comprovadas."
    ),
)

add_check(
    "S00B_SEMANTIC_DUPLICATE",
    "STRUCTURAL",
    (
        "Igualdade semântica entre n_failed e event_FAIL_count "
        "confirmada diretamente no Parquet model-facing FULL."
    ),
    {
        "rows_checked": int(len(provenance_df)),
        "mismatch_count": provenance_mismatch_count,
        "semantic_duplicate_verified": semantic_duplicate_verified,
    },
    {
        "mismatch_count": 0,
        "semantic_duplicate_verified": True,
    },
    "PASS" if semantic_duplicate_verified else "FAIL",
    (
        "window_5min_series_allcells_model_facing_000000000000.parquet"
        ":n_failed,event_FAIL_count"
    ),
    (
        "Interromper a auditoria se houver qualquer divergência entre "
        "n_failed e event_FAIL_count no artefato model-facing."
    ),
)


# ---------- STRUCTURAL ----------

observed_cells = sorted(
    full_summary["cell_id"].dropna().unique().tolist()
)
add_check(
    "S01",
    "STRUCTURAL",
    "Células observadas no NB11_FULL são a–h.",
    observed_cells,
    EXPECTED_CELLS,
    "PASS" if observed_cells == EXPECTED_CELLS else "FAIL",
    "full_metrics_summary:cell_id",
    "Investigar perda, duplicação ou mistura de fontes.",
)

false_cells = sorted(
    modelability_summary.loc[
        ~normalize_bool(
            modelability_summary["is_modelable"]
        ),
        "cell_id",
    ].tolist()
)
add_check(
    "S03",
    "STRUCTURAL",
    "A única célula com is_modelable=False é d.",
    false_cells,
    ["d"],
    "PASS" if false_cells == ["d"] else "FAIL",
    "AUD02_modelability_summary.csv:is_modelable",
    "Investigar o gate sem substituir o campo por rótulo.",
)

winner_counts = (
    nominal_winners.groupby("cell_id").size().to_dict()
)
one_winner_each = (
    len(winner_counts) > 0
    and all(count == 1 for count in winner_counts.values())
)
add_check(
    "S04",
    "STRUCTURAL",
    "A regra soberana gera um vencedor por célula modelável.",
    winner_counts,
    "1 por célula modelável",
    "PASS" if one_winner_each else "FAIL",
    "AUD02_nominal_winners_by_cell.csv",
    "Revisar ordenação, desempates ou duplicidades.",
)

add_check(
    "S06",
    "STRUCTURAL",
    (
        "AP(score_raw) e AP(score_calibrated) "
        "preservam igualdade nos resultados soberanos."
    ),
    {
        "all_equal": ap_raw_calibrated_equal_all,
        "by_cell": (
            pr_auc_lift_fixed_test[
                [
                    "cell_id",
                    "average_precision_score_raw",
                    "average_precision_score_calibrated",
                    "ap_raw_equals_calibrated",
                ]
            ].to_dict(orient="records")
        ),
    },
    {
        "all_equal": True,
        "interpretation": (
            "resultado observado; não pressuposto geral "
            "sobre qualquer método de calibração"
        ),
    },
    (
        "PASS"
        if ap_raw_calibrated_equal_all
        else "REVIEW"
    ),
    "AUD02_pr_auc_lift_fixed_test_by_cell.csv",
    (
        "Se houver divergência, verificar mudança de ordenação "
        "pela calibração; o lift oficial continua baseado "
        "no score_calibrated."
    ),
)

# S15 é incluído após todas as saídas.

# ---------- REGRESSION ----------

expected_values = expected_snapshot["values"]
n_modelable = int(
    nominal_winners["cell_id"].nunique()
)

add_check(
    "R02",
    "REGRESSION",
    "Quantidade de células modeláveis versus snapshot.",
    n_modelable,
    expected_values["n_modelable_cells"],
    (
        "PASS"
        if n_modelable
        == expected_values["n_modelable_cells"]
        else "REVIEW"
    ),
    "AUD02_nominal_winners_by_cell.csv",
    "Investigar; atualizar snapshot só com justificativa.",
)

observed_frequency = {
    row["model"]: int(row["n_cells"])
    for _, row in model_frequency.iterrows()
}
add_check(
    "R05",
    "REGRESSION",
    "Frequência empírica dos modelos soberanos.",
    observed_frequency,
    expected_values["model_frequency"],
    (
        "PASS"
        if observed_frequency
        == expected_values["model_frequency"]
        else "REVIEW"
    ),
    "AUD02_model_frequency.csv",
    "Mudança não implica erro automático.",
)

observed_roc_min = float(
    kaggle_comparison["roc_auc_tscv_mean"].min()
)
observed_roc_max = float(
    kaggle_comparison["roc_auc_tscv_mean"].max()
)

add_check(
    "R06",
    "REGRESSION",
    "ROC-AUC mínimo soberano.",
    observed_roc_min,
    expected_values["roc_auc_min"],
    (
        "PASS"
        if np.isclose(
            observed_roc_min,
            expected_values["roc_auc_min"],
            atol=1e-12,
        )
        else "REVIEW"
    ),
    "AUD02_kaggle_comparison.csv",
    "Investigar vencedor, cenário ou métrica.",
)
add_check(
    "R07",
    "REGRESSION",
    "ROC-AUC máximo soberano.",
    observed_roc_max,
    expected_values["roc_auc_max"],
    (
        "PASS"
        if np.isclose(
            observed_roc_max,
            expected_values["roc_auc_max"],
            atol=1e-12,
        )
        else "REVIEW"
    ),
    "AUD02_kaggle_comparison.csv",
    "Investigar vencedor, cenário ou métrica.",
)

n_above = int(
    kaggle_comparison["supera_referencia"].sum()
)
not_above_cells = sorted(
    kaggle_comparison.loc[
        ~kaggle_comparison["supera_referencia"],
        "cell_id",
    ].tolist()
)
excluded_cells = sorted(
    set(EXPECTED_CELLS)
    - set(kaggle_comparison["cell_id"])
)

add_check(
    "R08",
    "REGRESSION",
    "Quantidade acima do valor de referência.",
    n_above,
    expected_values["n_above_reference"],
    (
        "PASS"
        if n_above
        == expected_values["n_above_reference"]
        else "REVIEW"
    ),
    "AUD02_kaggle_comparison.csv",
    "Investigar; não forçar retorno ao snapshot.",
)
add_check(
    "R09",
    "REGRESSION",
    "Células que não superam a referência.",
    not_above_cells,
    expected_values["not_above_reference_cells"],
    (
        "PASS"
        if not_above_cells
        == expected_values["not_above_reference_cells"]
        else "REVIEW"
    ),
    "AUD02_kaggle_comparison.csv",
    "Investigar e documentar a causa.",
)
add_check(
    "R10",
    "REGRESSION",
    "Células ausentes da comparação.",
    excluded_cells,
    expected_values[
        "excluded_from_reference_comparison"
    ],
    (
        "PASS"
        if excluded_cells
        == expected_values[
            "excluded_from_reference_comparison"
        ]
        else "REVIEW"
    ),
    "AUD02_kaggle_comparison.csv",
    "Verificar se decorre do gate de modelabilidade.",
)

# ---------- TRACEABILITY ----------

lift_has_sources = bool(
    "full_scores_calibrated"
    in SOURCE_REGISTRY
    and "nb14_scenario_summary"
    in SOURCE_REGISTRY
    and "full_metrics_summary"
    in SOURCE_REGISTRY
)

mixed_lift_protocol = False

lift_traceability_complete = bool(
    lift_has_sources
    and official_lift_valid
)

lift_status = (
    "PASS"
    if lift_traceability_complete
    else "FAIL"
)

lift_action = (
    "Nenhuma."
    if lift_traceability_complete
    else (
        "Revisar seleção soberana, split fixo, "
        "contagens, prevalência e fonte primária."
    )
)

add_check(
    "T11",
    "TRACEABILITY",
    (
        "Lift oficial = AP/prevalência nas mesmas "
        "linhas do teste fixo herdado."
    ),
    {
        "all_terms_traced": lift_has_sources,
        "official_lift_valid": (
            official_lift_valid
        ),
        "mixed_protocol": False,
        "protocol": (
            "ALIGNED_NB11_FIXED_INHERITED_TEST"
        ),
        "n_cells": len(
            pr_auc_lift_fixed_test
        ),
        "row_checks": {
            column: bool(
                pr_auc_lift_fixed_test[
                    column
                ].all()
            )
            for column in official_lift_row_checks
        },
    },
    {
        "formula": (
            "average_precision_score("
            "y_true,score_calibrated"
            ")/mean(y_true)"
        ),
        "same_observations": True,
        "n_cells": len(modelable_cell_ids),
        "primary_source": "NB11_FULL",
    },
    lift_status,
    (
        "AUD02_pr_auc_lift_fixed_test_by_cell.csv | "
        "AUD02_lift_protocol_decision.json"
    ),
    lift_action,
)

anticipation_traced = bool(
    anticipation_claim_audit.iloc[0][
        "all_cell_rates_recomputed"
    ]
    and anticipation_claim_audit.iloc[0][
        "n_cells"
    ] == n_modelable
)
add_check(
    "T12",
    "TRACEABILITY",
    "Antecipação possui tau, numerador e denominador.",
    {
        "median": anticipation_median,
        "n_cells": int(
            anticipation_claim_audit.iloc[0]["n_cells"]
        ),
        "rates_recomputed": anticipation_traced,
    },
    "evidência completa nas células modeláveis",
    "PASS" if anticipation_traced else "FAIL",
    "AUD02_anticipation_claim_audit.csv",
    "Não citar 33% sem resolver denominadores.",
)

required_governance_numeric_columns = [
    "lstm_f1",
    "delta_f1",
    "lstm_f1_se",
    "delta_f1_se_ratio",
]

required_governance_text_columns = [
    "governance_decision",
    "recommended_score_source",
    "score_source_for_nb14",
]

required_governance_evidence_columns = [
    "lstm_f1_evidence",
    "delta_f1_evidence",
    "lstm_f1_se_evidence",
    "ratio_evidence",
    "decision_evidence",
    "recommendation_evidence",
    "actual_source_evidence",
]

numeric_trace_complete = (
    len(model_governance) > 0
    and model_governance[
        required_governance_numeric_columns
    ].notna().all().all()
)

text_trace_complete = (
    len(model_governance) > 0
    and model_governance[
        required_governance_text_columns
    ].notna().all().all()
)

evidence_trace_complete = (
    len(model_governance) > 0
    and all(
        model_governance[column]
        .astype(str)
        .ne("None:None")
        .all()
        for column in required_governance_evidence_columns
    )
)

ratio_semantics_complete = (
    len(model_governance) > 0
    and (
        model_governance["ratio_semantics"]
        == (
            "heuristica_descritiva_nao_pareada_"
            "baseada_no_se_da_lstm"
        )
    ).all()
    and (
        model_governance["comparison_pairing"]
        == "nao_pareada_contra_nb11_full"
    ).all()
    and (
        model_governance["ratio_denominator"]
        == "lstm_f1_se"
    ).all()
)

governance_trace_complete = all([
    numeric_trace_complete,
    text_trace_complete,
    evidence_trace_complete,
    ratio_semantics_complete,
])

add_check(
    "T13",
    "TRACEABILITY",
    (
        "Governança LSTM rastreada integralmente e razão "
        "classificada como heurística descritiva não pareada."
    ),
    {
        "rows": len(model_governance),
        "numeric_trace_complete": (
            numeric_trace_complete
        ),
        "text_trace_complete": text_trace_complete,
        "evidence_trace_complete": (
            evidence_trace_complete
        ),
        "ratio_semantics_complete": (
            ratio_semantics_complete
        ),
    },
    {
        "all_modelable_cells_traced": True,
        "ratio_semantics": (
            "heuristica_descritiva_nao_pareada_"
            "baseada_no_se_da_lstm"
        ),
    },
    "PASS" if governance_trace_complete else "FAIL",
    "AUD02_model_governance_by_cell.csv",
    (
        "Completar valores, aliases e evidências; "
        "não usar significativo nem pareado."
    ),
)

coherence_counts = (
    model_governance["score_source_coherence"]
    .value_counts(dropna=False)
    .to_dict()
)

all_coherent = (
    len(model_governance) > 0
    and (
        model_governance["score_source_coherence"]
        == "COHERENT"
    ).all()
)

no_lstm_primary_promotion = (
    promoted_lstm_primary_cells == []
)

secondary_evaluation_complete = (
    secondary_lstm_evaluation_cells
    == modelable_cell_ids
)

gate_structure_consistent = (
    robustness_gate_cells == []
    and promoted_lstm_primary_cells == []
)

lstm_governance_consistent = all([
    all_coherent,
    no_lstm_primary_promotion,
    secondary_evaluation_complete,
    gate_structure_consistent,
])

add_check(
    "T14",
    "TRACEABILITY",
    (
        "Governança posterior da LSTM distingue gate nominal, "
        "gate de robustez, promoção primária e avaliação secundária."
    ),
    {
        "coherence_counts": coherence_counts,
        "nominal_delta_gate_cells": (
            nominal_delta_gate_cells
        ),
        "robustness_gate_cells": (
            robustness_gate_cells
        ),
        "promoted_lstm_primary_cells": (
            promoted_lstm_primary_cells
        ),
        "secondary_lstm_evaluation_cells": (
            secondary_lstm_evaluation_cells
        ),
    },
    {
        "COHERENT": len(model_governance),
        "nominal_delta_gate_cells": (
            "derivado dos artefatos vigentes"
        ),
        "robustness_gate_cells": [],
        "promoted_lstm_primary_cells": [],
        "secondary_lstm_evaluation_cells": (
            modelable_cell_ids
        ),
        "primary_source": "NB11_FULL",
    },
    "PASS" if lstm_governance_consistent else "FAIL",
    (
        "AUD02_model_governance_by_cell.csv | "
        "AUD02_lstm_governance_summary.json"
    ),
    (
        "Revisar separadamente o primeiro gate nominal, "
        "o gate de robustez, a fonte primária e as linhas "
        "secundárias do NB14_FULL."
    ),
)

if RUN_STARTED_AT.date() <= FREEZE_DATE:
    freeze_status = "PASS"
    freeze_observed = "executado_antes_do_congelamento"
else:
    freeze_status = (
        "PASS"
        if GOVERNANCE_EXCEPTION_PATH.exists()
        else "FAIL"
    )
    freeze_observed = (
        str(GOVERNANCE_EXCEPTION_PATH)
        if GOVERNANCE_EXCEPTION_PATH.exists()
        else "excecao_ausente"
    )

add_check(
    "T15",
    "TRACEABILITY",
    "AUD02 respeita congelamento ou exceção explícita.",
    freeze_observed,
    (
        f"até {FREEZE_DATE.isoformat()} "
        "ou exceção registrada"
    ),
    freeze_status,
    (
        "AUD02_governance_exception.json "
        "quando aplicável"
    ),
    "Registrar exceção se executado após congelamento.",
)


# ==============================================================================
# NOTA/SEÇÃO DOCUMENTAL 13
# ==============================================================================
# Consolidação da matriz de afirmações


# ============================================================
# Consolidação das afirmações
# ============================================================

hash_by_key = {
    key: SOURCE_FINGERPRINTS_BEFORE[key]["sha256"]
    for key in SOURCE_FINGERPRINTS_BEFORE
}

def claim_row(
    claim_id: str,
    status: str,
    result: str,
    evidence_files: list[str],
    evidence_fields: list[str],
    derivation_rule: str,
    notes: str,
) -> dict[str, Any]:
    hashes = []
    for key, path in SOURCE_REGISTRY.items():
        if str(path) in evidence_files:
            hashes.append(hash_by_key[key])

    return {
        "claim_id": claim_id,
        "status": status,
        "result": result,
        "evidence_files": " | ".join(evidence_files),
        "evidence_fields": " | ".join(evidence_fields),
        "derivation_rule": derivation_rule,
        "source_hashes_sha256": " | ".join(hashes),
        "notes": notes,
    }

claims_results = []

claims_results.append(claim_row(
    "4.1",
    "CONFIRMED",
    (
        f"{len(observed_cells)} células observadas; "
        f"{n_modelable} com is_modelable=True; "
        f"is_modelable=False em {false_cells}."
    ),
    [
        str(SOURCE_REGISTRY["full_metrics_summary"]),
        str(SOURCE_REGISTRY["full_modelability_overview"]),
        str(SOURCE_REGISTRY["nb14_scenario_summary"]),
    ],
    [
        "cell_id",
        "is_modelable/status_modelagem",
        "n_positivos_train",
        "n_positivos_test",
        "n_evaluable_episodes_tau_reference",
    ],
    "contagem e filtro por célula/cenário",
    "Unidades janela e episódio permanecem separadas.",
))

claims_results.append(claim_row(
    "4.2",
    "CONFIRMED",
    (
        f"ROC-AUC soberano entre "
        f"{observed_roc_min:.6f} e "
        f"{observed_roc_max:.6f}."
    ),
    [str(SOURCE_REGISTRY["full_metrics_summary"])],
    ["roc_auc_tscv_mean"],
    "min/max após seleção soberana por F1/recall/AP",
    "Faixa exclusiva das células modeláveis.",
))

lift_claim_status = (
    "CONFIRMED"
    if official_lift_valid
    else "UNRESOLVED"
)

claims_results.append(claim_row(
    "4.3",
    lift_claim_status,
    (
        "No teste fixo herdado, o lift de PR-AUC "
        f"sobre prevalência varia de {lift_min:.3f}× "
        f"a {lift_max:.3f}× nas sete células modeláveis."
        if official_lift_valid
        else "Lift alinhado não integralmente reproduzido."
    ),
    [
        str(
            SOURCE_REGISTRY[
                "full_scores_calibrated"
            ]
        ),
        str(
            SOURCE_REGISTRY[
                "full_metrics_summary"
            ]
        ),
        str(
            SOURCE_REGISTRY[
                "nb14_scenario_summary"
            ]
        ),
    ],
    [
        "y_true",
        "score_calibrated",
        "validation=fixed_inherited_80_20",
        "n_scored_rows",
        "positive_rate_tau_reference",
        "is_primary_score_source",
    ],
    (
        "average_precision_score("
        "y_true, score_calibrated"
        ") / mean(y_true)"
    ),
    (
        "Protocolo oficial alinhado no teste fixo. "
        "O TSCV permanece como diagnóstico complementar; "
        "a antiga razão mista é apenas legado de rastreabilidade."
    ),
))

claims_results.append(claim_row(
    "4.4",
    (
        "CONFIRMED"
        if anticipation_traced
        else "UNRESOLVED"
    ),
    (
        "Mediana entre células modeláveis = "
        f"{100 * anticipation_median:.2f}%."
    ),
    [str(SOURCE_REGISTRY["nb14_scenario_summary"])],
    [
        "tau_reference",
        "n_evaluable_episodes_tau_reference",
        "n_anticipated_episodes_tau_reference",
        "episode_anticipation_rate_tau_reference",
    ],
    (
        "mediana das taxas por célula; "
        "taxa = antecipados/avaliáveis"
    ),
    "Episódios não avaliáveis não entram no denominador.",
))

refined_model = canonical_model_name(
    refined_winner.get("model")
    or refined_winner.get("winner_model")
    or refined_winner.get("modelo")
)
claims_results.append(claim_row(
    "4.5",
    (
        "CONFIRMED"
        if refined_model == "logistic_regression"
        else "REVIEW"
    ),
    (
        "Modelo vencedor do protótipo refinado: "
        f"{refined_model}."
    ),
    [str(SOURCE_REGISTRY["refined_winner"])],
    ["model/winner_model", benchmark_field],
    "leitura direta do JSON do vencedor",
    "Preservar protocolo e cenário na comparação.",
))

claims_results.append(claim_row(
    "4.6",
    "CONFIRMED",
    (
        "Frequência soberana por modelo: "
        f"{observed_frequency}."
    ),
    [str(SOURCE_REGISTRY["full_metrics_summary"])],
    [
        "model",
        "f1_tscv_mean",
        "recall_tscv_mean",
        "average_precision_tscv_mean",
    ],
    "seleção soberana e contagem por modelo",
    (
        "A governança posterior está em "
        "AUD02_model_governance_by_cell.csv."
    ),
))

claims_results.append(claim_row(
    "4.7",
    (
        "CONFIRMED"
        if governance_trace_complete
        and lstm_governance_consistent
        else "PARTIAL"
    ),
    (
        "Nenhuma LSTM foi promovida como fonte primária. "
        f"Gate nominal ΔF1 >= 0,03: {nominal_delta_gate_cells}; "
        f"gate de robustez ΔF1/SE >= 2: {robustness_gate_cells}; "
        f"promoção primária: {promoted_lstm_primary_cells}; "
        "avaliação secundária posterior no NB14_FULL: "
        f"{secondary_lstm_evaluation_cells}. "
        "As fontes NB13_FULL e NB13a_FULL foram preservadas "
        "somente como sensibilidades secundárias."
    ),
    [
        str(SOURCE_REGISTRY[key])
        for key in [
            "nb13_decision",
            "nb13_uncertainty",
            "nb13a_decision",
            "nb13a_delta_nb11",
            "nb14_scenario_summary",
        ]
        if key in SOURCE_REGISTRY
    ],
    [
        "decision",
        "nb14_score_source_recommendation",
        "delta_f1_tuned_minus_nb11/"
        "delta_f1_lstm_minus_nb11",
        "delta_over_lstm_tuned_f1_se/"
        "delta_over_lstm_f1_se",
        "score_source",
        "is_primary_score_source",
    ],
    (
        "distinguir: gate nominal, gate de robustez, "
        "promoção efetiva e avaliação secundária no NB14_FULL"
    ),
    (
        f"Gate nominal observado na linhagem vigente: {nominal_delta_gate_cells}. "
        "Nenhuma célula atingiu o gate de robustez suficiente para promoção. "
        "A avaliação posterior da LSTM permaneceu secundária."
    ),
))

claims_results.append(claim_row(
    "4.8",
    "CONTEXT_ONLY",
    (
        "Brier preservado apenas como métrica "
        "registrada; sem nova calibração."
    ),
    [str(SOURCE_REGISTRY["full_metrics_summary"])],
    ["brier_tscv_mean quando disponível"],
    "registro documental",
    "Não reabre o escopo experimental.",
))

claims_results.append(claim_row(
    "4.9",
    "CONFIRMED",
    (
        f"{n_above} de {n_modelable} acima da referência "
        f"{KAGGLE_REFERENCE_ROC_AUC:.12f}; "
        f"não acima: {not_above_cells}; "
        f"excluídas: {excluded_cells}."
    ),
    [
        str(SOURCE_REGISTRY["refined_winner"]),
        str(SOURCE_REGISTRY["full_metrics_summary"]),
    ],
    [benchmark_field, "roc_auc_tscv_mean"],
    (
        "seleção soberana por F1 e comparação "
        "ROC-AUC > referência"
    ),
    "Comparação descritiva, sem teste entre bases.",
))

claims_audit = claims_spec.merge(
    pd.DataFrame(claims_results),
    on="claim_id",
    how="left",
    validate="one_to_one",
)

PATH_CLAIMS_CSV = write_csv(
    claims_audit,
    "AUD02_claims_audit.csv",
)
PATH_CLAIMS_JSON = write_json(
    claims_audit.to_dict(orient="records"),
    "AUD02_claims_audit.json",
)

report_lines = [
    "# AUD02 — Relatório de afirmações",
    "",
    f"Gerado em: {RUN_STARTED_AT.isoformat()}",
    "",
]

for _, row in claims_audit.iterrows():
    report_lines.extend([
        f"## {row['claim_id']} — {row['status']}",
        "",
        f"**Afirmação:** {row['claim_text']}",
        "",
        f"**Resultado:** {row['result']}",
        "",
        f"**Regra:** {row['derivation_rule']}",
        "",
        f"**Arquivos:** `{row['evidence_files']}`",
        "",
        f"**Campos:** `{row['evidence_fields']}`",
        "",
        f"**Notas:** {row['notes']}",
        "",
    ])

PATH_CLAIMS_REPORT = write_text(
    "\n".join(report_lines),
    "AUD02_claims_report.md",
)

display(claims_audit)


# ============================================================
# S15 — fontes oficiais inalteradas
# ============================================================

SOURCE_FINGERPRINTS_AFTER = {
    key: fingerprint(path)
    for key, path in SOURCE_REGISTRY.items()
}

changed_sources = []

for key in SOURCE_FINGERPRINTS_BEFORE:
    before = SOURCE_FINGERPRINTS_BEFORE[key]
    after = SOURCE_FINGERPRINTS_AFTER[key]

    if (
        before["size_bytes"] != after["size_bytes"]
        or before["mtime_ns"] != after["mtime_ns"]
        or before["sha256"] != after["sha256"]
    ):
        changed_sources.append({
            "source_key": key,
            "before": before,
            "after": after,
        })

all_outputs_inside_audit_dir = all(
    is_under(path, AUDIT_DIR)
    for path in AUDIT_DIR.iterdir()
    if path.is_file()
)

s15_pass = (
    len(changed_sources) == 0
    and all_outputs_inside_audit_dir
)

add_check(
    "S15",
    "STRUCTURAL",
    (
        "AUD02 não modifica arquivos oficiais e "
        "escreve somente no AUDIT_DIR."
    ),
    {
        "changed_sources": changed_sources,
        "all_outputs_inside_audit_dir": (
            all_outputs_inside_audit_dir
        ),
    },
    {
        "changed_sources": [],
        "all_outputs_inside_audit_dir": True,
    },
    "PASS" if s15_pass else "FAIL",
    "SHA-256/mtime/size antes e depois",
    "Interromper e investigar qualquer escrita indevida.",
)

checks_df = pd.DataFrame(checks)
PATH_CHECKS = write_csv(
    checks_df,
    "AUD02_checks.csv",
)

display(checks_df)


# ============================================================
# Resumo, conclusão e manifesto final
# ============================================================

structural_failures = checks_df[
    (checks_df["check_class"] == "STRUCTURAL")
    & (checks_df["status"] == "FAIL")
]
structural_reviews = checks_df[
    (checks_df["check_class"] == "STRUCTURAL")
    & (checks_df["status"] == "REVIEW")
]
regression_reviews = checks_df[
    (checks_df["check_class"] == "REGRESSION")
    & (checks_df["status"] == "REVIEW")
]
traceability_failures = checks_df[
    (checks_df["check_class"] == "TRACEABILITY")
    & (checks_df["status"] == "FAIL")
]

methodological_qualifications = checks_df[
    (checks_df["check_class"] == "TRACEABILITY")
    & (checks_df["status"] == "REVIEW")
]

if not structural_failures.empty:
    overall_status = "FAIL_STRUCTURAL"
elif not traceability_failures.empty:
    overall_status = "FAIL_TRACEABILITY"
elif not structural_reviews.empty:
    overall_status = "PASS_WITH_STRUCTURAL_REVIEW"
elif not regression_reviews.empty:
    overall_status = "PASS_WITH_REGRESSION_REVIEW"
elif not methodological_qualifications.empty:
    overall_status = "PASS_WITH_METHODOLOGICAL_QUALIFICATION"
else:
    overall_status = "PASS"

methodological_cautions = [
    (
        "A comparação do item 4.9 permanece descritiva "
        "entre bases, protocolos e cenários não idênticos."
    ),
    (
        "Os cenários soberanos variam entre células; "
        "não há confronto controlado isolando base e cenário."
    ),
    (
        "A margem da célula f frente ao valor de referência "
        "não possui desenho inferencial."
    ),
    (
        "Seleção e ROC-AUC do item 4.9 permanecem no TSCV; "
        "o lift oficial usa o teste fixo herdado. Trata-se de "
        "protocolos diferentes por finalidade da métrica, "
        "sem mistura dentro de um mesmo cálculo."
    ),
    (
        "PASS significa ausência de falhas nos checks definidos, "
        "não ausência de limitações metodológicas da dissertação."
    ),
]

summary_payload = {
    "notebook": (
        "AUD02_auditoria_integrada_evidencias.ipynb"
    ),
    "audit_role": "governanca_somente_leitura",
    "experimental_branch_order": (
        "NB04_FULL-NB15_FULL -> NB16_FULL; AUD02 posterior e independente, auditando somente fontes ate NB15_FULL"
    ),
    "audit_position": (
        "AUD02 transversal, executado após o fechamento experimental, sem consumir NB16_FULL"
    ),
    "nb16_action": "PRESERVED_NOT_CONSUMED",
    "nb16_consumed": False,
    "generated_at": (
        datetime.now(timezone.utc)
        .astimezone().isoformat()
    ),
    "overall_status": overall_status,
    "snapshot_status": SNAPSHOT_STATUS,
    "snapshot_id": expected_snapshot["snapshot_id"],
    "freeze_date": FREEZE_DATE.isoformat(),
    "output_directory": str(AUDIT_DIR),
    "counts": {
        "sources": len(SOURCE_REGISTRY),
        "claims": len(claims_audit),
        "checks": len(checks_df),
        "structural_failures": len(
            structural_failures
        ),
        "structural_reviews": len(
            structural_reviews
        ),
        "regression_reviews": len(
            regression_reviews
        ),
        "traceability_failures": len(
            traceability_failures
        ),
        "methodological_qualifications": len(
            methodological_qualifications
        ),
    },
    "headline_results": {
        "observed_cells": observed_cells,
        "modelable_cells": sorted(
            nominal_winners["cell_id"].tolist()
        ),
        "is_modelable_false_cells": false_cells,
        "model_frequency": observed_frequency,
        "roc_auc_min": observed_roc_min,
        "roc_auc_max": observed_roc_max,
        "reference_roc_auc": KAGGLE_REFERENCE_ROC_AUC,
        "n_above_reference": n_above,
        "not_above_reference_cells": (
            not_above_cells
        ),
        "anticipation_median": (
            anticipation_median
        ),
        "lift_min": float(lift_min),
        "lift_max": float(lift_max),
        "lift_protocols": (
            pr_auc_lift[
                "protocol_alignment"
            ].value_counts(
                dropna=False
            ).to_dict()
        ),
        "lift_official_protocol": (
            "ALIGNED_NB11_FIXED_INHERITED_TEST"
        ),
        "lift_legacy_mixed_min": (
            legacy_lift_min
        ),
        "lift_legacy_mixed_max": (
            legacy_lift_max
        ),
        "lift_tscv_diagnostic": {
            "role": "complementary_not_official",
            "ratio_of_means_min": float(
                pr_auc_lift_tscv_diagnostic[
                    "lift_ratio_of_means"
                ].min()
            ),
            "ratio_of_means_max": float(
                pr_auc_lift_tscv_diagnostic[
                    "lift_ratio_of_means"
                ].max()
            ),
            "aggregation_ambiguity_documented": True,
        },
        "lstm_governance": {
            "nominal_delta_gate_cells": (
                nominal_delta_gate_cells
            ),
            "robustness_gate_cells": (
                robustness_gate_cells
            ),
            "promoted_lstm_primary_cells": (
                promoted_lstm_primary_cells
            ),
            "secondary_lstm_evaluation_cells": (
                secondary_lstm_evaluation_cells
            ),
            "final_position": (
                "no_primary_promotion_secondary_only"
            ),
        },
    },
    "structural_failures": (
        structural_failures.to_dict(
            orient="records"
        )
    ),
    "structural_reviews": (
        structural_reviews.to_dict(
            orient="records"
        )
    ),
    "regression_reviews": (
        regression_reviews.to_dict(
            orient="records"
        )
    ),
    "traceability_failures": (
        traceability_failures.to_dict(
            orient="records"
        )
    ),
    "methodological_qualifications": (
        methodological_qualifications.to_dict(
            orient="records"
        )
    ),
    "status_interpretation": (
        "PASS indica aprovação dos checks definidos; "
        "não elimina cautelas metodológicas em prosa."
    ),
    "methodological_cautions": methodological_cautions,
    "next_step": (
        "Executar DOC01 após aprovação do AUD02 para regenerar a Matriz documental; "
        "depois propagar seletivamente as mudanças ao Mapa da Escrita. "
        "NB16_FULL permanece preservado e não é consumido pela auditoria."
    ),
}

PATH_SUMMARY = write_json(
    summary_payload,
    "AUD02_summary.json",
)

conclusion_lines = [
    "# AUD02 — Notas de conclusão",
    "",
    f"**Status geral:** `{overall_status}`",
    "",
    "## Escopo",
    "",
    "- Auditoria somente-leitura.",
    "- Fontes: protótipo refinado e NB04_FULL–NB15_FULL.",
    "- NB16_FULL não foi consumido.",
    "- Nenhum treinamento ou reotimização.",
    "",
    "## Resultados centrais",
    "",
    f"- Células observadas: {observed_cells}.",
    (
        "- Células modeláveis: "
        f"{sorted(nominal_winners['cell_id'].tolist())}."
    ),
    (
        "- Única célula com `is_modelable=False`: "
        f"{false_cells}."
    ),
    (
        "- Frequência soberana por modelo: "
        f"{observed_frequency}."
    ),
    (
        "- ROC-AUC soberano: "
        f"{observed_roc_min:.6f} a "
        f"{observed_roc_max:.6f}."
    ),
    (
        "- Comparação descritiva: "
        f"{n_above}/{n_modelable} acima; "
        f"não acima: {not_above_cells}."
    ),
    (
        "- Mediana de antecipação: "
        f"{100 * anticipation_median:.2f}%."
    ),
    (
        "- Lift oficial no teste fixo herdado: "
        f"{lift_min:.3f}× a {lift_max:.3f}×."
    ),
    (
        "- Faixa legada sob protocolo misto, preservada "
        "apenas para rastreabilidade: "
        f"{legacy_lift_min:.3f}× a {legacy_lift_max:.3f}×."
    ),
    (
        "- AP(score_raw) = AP(score_calibrated) nas sete "
        f"células soberanas: {ap_raw_calibrated_equal_all}."
    ),
    (
        "- LSTM — gate nominal ΔF1 >= 0,03: "
        f"{nominal_delta_gate_cells}; gate de robustez "
        f"ΔF1/SE >= 2: {robustness_gate_cells}."
    ),
    (
        "- LSTM — promoção como fonte primária: "
        f"{promoted_lstm_primary_cells}; avaliação secundária "
        "no NB14_FULL: "
        f"{secondary_lstm_evaluation_cells}."
    ),
    (
        "- Posição final da LSTM: nenhuma promoção; "
        "NB13_FULL e NB13a_FULL permaneceram apenas como "
        "sensibilidades secundárias."
    ),
    "",
    "## Ressalvas",
    "",
    "- Kaggle × FULL não é confronto controlado.",
    "- Dispersão entre folds não é teste entre bases.",
    (
        "- `ΔF1/SE` é "
        "`heuristica_descritiva_nao_pareada_baseada_no_se_da_lstm`."
    ),
    (
        "- Política de protocolos: seleção e comparação de "
        "ROC-AUC no TSCV; lift e métricas operacionais no "
        "teste fixo herdado."
    ),
    (
        "- O status PASS aprova os checks definidos, mas não "
        "remove as cautelas metodológicas registradas."
    ),
    "",
    "## Próxima etapa",
    "",
    (
        "Após aprovação do AUD02, executar DOC01 para regenerar a Matriz; "
        "em seguida atualizar seletivamente o Mapa da Escrita. "
        "NB16_FULL permanece preservado e não foi consumido."
    ),
]

PATH_CONCLUSION = write_text(
    "\n".join(conclusion_lines),
    "AUD02_conclusion_notes.md",
)

manifest_rows = []

for path in sorted(
    AUDIT_DIR.iterdir(),
    key=lambda p: p.name,
):
    if not path.is_file():
        continue
    if (
        path.name
        == "AUD02_artifact_manifest_sha256.csv"
    ):
        continue

    manifest_rows.append({
        "filename": path.name,
        "path": str(path.resolve()),
        "size_bytes": int(path.stat().st_size),
        "sha256": sha256_file(path),
    })

manifest_df = pd.DataFrame(manifest_rows)
PATH_MANIFEST = write_csv(
    manifest_df,
    "AUD02_artifact_manifest_sha256.csv",
)

print("=" * 100)
print("AUD02 CONCLUÍDO")
print("=" * 100)
print("Status geral :", overall_status)
print("Saída        :", AUDIT_DIR)
print("Manifesto    :", PATH_MANIFEST)
print("NB16 lido?   :", False)
print(
    "Próxima etapa: após aprovação do AUD02, executar DOC01; "
    "depois atualizar seletivamente o Mapa da Escrita. "
    "NB16_FULL permanece preservado e não foi consumido."
)

if (
    STRICT_STRUCTURAL_FAILURE
    and not structural_failures.empty
):
    raise AssertionError(
        "Falha estrutural. Artefatos foram "
        "gravados para diagnóstico. Checks: "
        f"{structural_failures['check_id'].tolist()}"
    )


# ==============================================================================
# NOTA/SEÇÃO DOCUMENTAL 14
# ==============================================================================
# Artefatos produzidos
# text
# AUD02_source_inventory.csv
# AUD02_source_hashes.csv
# AUD02_schema_checks.csv
# AUD02_model_taxonomy.csv
# AUD02_modelability_by_cell_scenario.csv
# AUD02_modelability_summary.csv
# AUD02_baseline_ranking_by_cell_scenario.csv
# AUD02_nominal_winners_by_cell.csv
# AUD02_model_frequency.csv
# AUD02_model_governance_by_cell.csv
# AUD02_lstm_governance_summary.json
# AUD02_pr_auc_lift_by_cell.csv
# AUD02_pr_auc_lift_fixed_test_by_cell.csv
# AUD02_pr_auc_lift_mixed_legacy_by_cell.csv
# AUD02_pr_auc_lift_tscv_diagnostic_by_cell.csv
# AUD02_lift_protocol_decision.json
# AUD02_kaggle_comparison.csv
# AUD02_operational_metrics_by_cell.csv
# AUD02_anticipation_claim_audit.csv
# AUD02_claims_audit.csv
# AUD02_claims_audit.json
# AUD02_claims_report.md
# AUD02_checks.csv
# AUD02_expected_values.json
# AUD02_summary.json
# AUD02_conclusion_notes.md
# AUD02_artifact_manifest_sha256.csv
#
# Plano de ação associado
# - A4: permanece como entregável; atendida pelos Módulos 4 e 5.
# - A15: registrar a modelabilidade; atendida pelo Módulo 3.
# - A16: AUD02 concluído como auditoria transversal somente-leitura.
# - A17: refletir seletivamente as claims auditadas no Mapa da Escrita; manter NB11_FULL, NB14_FULL e NB16_FULL inalterados; o NB16_FULL não é fonte do AUD02.


Mounted at /content/drive
RUN_STARTED_AT : 2026-07-26T04:41:09.825703+00:00
DRIVE_ROOT     : /content/drive/MyDrive/Mestrado
FULL_ROOT      : /content/drive/MyDrive/Mestrado/04-reports/99_FULL_downstream
AUDIT_DIR      : /content/drive/MyDrive/Mestrado/04-reports/AUD02_evidence_audit
FREEZE_DATE    : 2026-07-31


,key,status,path,required,stage
0,refined_winner,FOUND_PREFERRED,/content/drive/MyDrive/Mestrado/04-reports/11_winner_model.json,True,PROTO_REFINADO
1,refined_metrics_summary,FOUND_PREFERRED,/content/drive/MyDrive/Mestrado/04-reports/11_metrics_summary.csv,False,PROTO_REFINADO
2,full_model_facing,FOUND_PREFERRED,/content/drive/MyDrive/Mestrado/02-datasets/99-full/03-model-facing/window_5min_series_allcells_model_facing_0000000...,True,FULL_DATASET
3,full_metrics_summary,FOUND_PREFERRED,/content/drive/MyDrive/Mestrado/04-reports/99_FULL_downstream/11_FULL_model_baselines/aggregate/11_FULL_metrics_summ...,True,NB11_FULL
4,full_metrics_tscv,FOUND_PREFERRED,/content/drive/MyDrive/Mestrado/04-reports/99_FULL_downstream/11_FULL_model_baselines/aggregate/11_FULL_metrics_tscv...,True,NB11_FULL
5,full_scores_calibrated,FOUND_PREFERRED,/content/drive/MyDrive/Mestrado/04-reports/99_FULL_downstream/11_FULL_model_baselines/aggregate/11_FULL_scores_calib...,True,NB11_FULL
6,full_modelability_overview,FOUND_PREFERRED,/content/drive/MyDrive/Mestrado/04-reports/99_FULL_downstream/11_FULL_model_baselines/aggregate/11_FULL_modelability...,True,NB11_FULL
7,full_feature_sets,FOUND_PREFERRED,/content/drive/MyDrive/Mestrado/04-reports/99_FULL_downstream/11_FULL_model_baselines/aggregate/11_FULL_feature_sets...,True,NB11_FULL
8,full_model_input_features,FOUND_PREFERRED,/content/drive/MyDrive/Mestrado/04-reports/99_FULL_downstream/11_FULL_model_baselines/aggregate/11_FULL_model_input_...,True,NB11_FULL
9,nb13_decision,FOUND_PREFERRED,/content/drive/MyDrive/Mestrado/04-reports/99_FULL_downstream/13_FULL_lstm/aggregate/13_FULL_lstm_decision_to_nb14_b...,False,NB13_FULL


,cell_id,scenario_label,feature_set,model,f1_tscv_mean,recall_tscv_mean,average_precision_tscv_mean,roc_auc_tscv_mean,roc_auc_fold_std,roc_auc_fold_min,roc_auc_fold_max,n_folds_valid,benchmark_roc_auc,delta_roc_auc,supera_referencia,comparison_semantics,inferential_test,status_modelagem,is_modelable,n_positivos_train,n_positivos_test
0,a,cell_a_W5_K24_H12_train_m2s_P1,temporal_core,logistic_regression,0.398095,0.580453,0.418901,0.767504,0.081540,0.714580,0.911372,5,0.647387,+0.120118,True,comparacao_descritiva_entre_bases_protocolos_e_cenarios_nao_identicos,nao_aplicavel,modelavel,True,890,144
1,b,cell_b_W5_K24_H12_train_m2s_P1,temporal_core,hist_gradient_boosting,0.496409,0.501855,0.485580,0.746623,0.104983,0.614385,0.897056,5,0.647387,+0.099236,True,comparacao_descritiva_entre_bases_protocolos_e_cenarios_nao_identicos,nao_aplicavel,modelavel,True,1196,147
2,c,cell_c_W5_K24_H6_train_m2s_P1,temporal_core,hist_gradient_boosting,0.401552,0.478661,0.392332,0.727005,0.088952,0.611913,0.836752,5,0.647387,+0.079618,True,comparacao_descritiva_entre_bases_protocolos_e_cenarios_nao_identicos,nao_aplicavel,modelavel,True,654,96
3,e,cell_e_W5_K12_H6_train_m2s_P1,temporal_core,hist_gradient_boosting,0.326230,0.416827,0.342667,0.716326,0.067619,0.654120,0.825689,5,0.647387,+0.068939,True,comparacao_descritiva_entre_bases_protocolos_e_cenarios_nao_identicos,nao_aplicavel,modelavel,True,414,76
4,f,cell_f_W5_K24_H12_train_m2s_P1,temporal_core,logistic_regression,0.385708,0.468899,0.409447,0.636979,0.102196,0.492747,0.741458,5,0.647387,-0.010408,False,comparacao_descritiva_entre_bases_protocolos_e_cenarios_nao_identicos,nao_aplicavel,modelavel,True,1049,231
5,g,cell_g_W5_K24_H12_train_m2s_P1,temporal_core,hist_gradient_boosting,0.633126,0.669714,0.662726,0.812583,0.064533,0.709517,0.876902,5,0.647387,+0.165196,True,comparacao_descritiva_entre_bases_protocolos_e_cenarios_nao_identicos,nao_aplicavel,modelavel,True,1640,490
6,h,cell_h_W5_K24_H12_train_m2s_P1,temporal_core,logistic_regression,0.516797,0.516274,0.549691,0.688894,0.042394,0.650494,0.751134,5,0.647387,+0.041508,True,comparacao_descritiva_entre_bases_protocolos_e_cenarios_nao_identicos,nao_aplicavel,modelavel,True,1729,384


Valor de referência: 0.647386784873
Acima: ['a', 'b', 'c', 'e', 'g', 'h']
Não acima: ['f']


,claim_id,claim_text,module,required_evidence,derivation_rule_spec,target_status
0,4.1,Existem oito células; sete têm is_modelable=True e a única com is_modelable=False é d.,M3,NB11_FULL modelability + suporte por cenário,contagem de cell_id e filtro por is_modelable,CONFIRMED
1,4.2,"O ROC-AUC soberano das células modeláveis varia aproximadamente de 0,637 a 0,813.",M7,NB11_FULL metrics summary,min/max após seleção soberana por F1/recall/AP,CONFIRMED
2,4.3,"No teste fixo herdado, o lift de PR-AUC sobre prevalência é calculado com numerador e denominador sobre exatamente a...",M6,escores NB11_FULL congelados + seleção primária NB14_FULL,"average_precision_score(y_true, score_calibrated) / mean(y_true)",CONFIRMED
3,4.4,A mediana da taxa de episódios antecipados no limiar de referência é aproximadamente 33%.,M8,"NB14_FULL: tau, numerador e denominador",mediana entre células modeláveis de antecipados/avaliáveis,CONFIRMED
4,4.5,A Regressão Logística é o vencedor do protótipo Kaggle refinado no cenário de referência.,M7/M9,11_winner_model.json,leitura direta do campo model,CONFIRMED
5,4.6,"No FULL, o resultado soberano não é dominado exclusivamente pela Regressão Logística.",M4/M5,ranking NB11_FULL + governança NB13/NB14,frequência e fonte efetiva de escores,CONFIRMED
6,4.7,Nenhuma LSTM foi promovida como fonte primária. As células que ultrapassam o piso nominal de ΔF1 são determinadas a ...,M5,NB13/NB13a/NB14,cruzamento por cell_id,CONFIRMED_OR_QUALIFIED
7,4.8,"Brier/calibração são contexto e limitação, sem reabrir o escopo.",M4/M9,métricas existentes e decisão documental,registro sem nova otimização,CONTEXT_ONLY
8,4.9,Seis das sete células modeláveis superam descritivamente a referência; f fica abaixo.,M7,AUD02_kaggle_comparison.csv,roc_auc_tscv_mean > referência,CONFIRMED


...

SNAPSHOT_STATUS: CREATED_INITIAL


,snapshot_id,created_at,basis,update_policy,values.n_modelable_cells,values.model_frequency.hist_gradient_boosting,values.model_frequency.logistic_regression,values.roc_auc_min,values.roc_auc_max,values.reference_roc_auc,values.n_above_reference,values.not_above_reference_cells,values.excluded_from_reference_comparison
0,AUD02-v1.2-corrected-lineage,2026-07-26,"Primeira auditoria formal da linhagem corrigida, após estabilização do pipeline. Os números governantes são derivado...",Não sobrescrever automaticamente após a primeira criação. Mudanças posteriores exigem novo snapshot e justificativa.,7,4,3,0.636979,0.812583,0.647387,6,[f],[]


,claim_id,claim_text,module,required_evidence,derivation_rule_spec,target_status,status,result,evidence_files,evidence_fields,derivation_rule,source_hashes_sha256,notes
0,4.1,Existem oito células; sete têm is_modelable=True e a única com is_modelable=False é d.,M3,NB11_FULL modelability + suporte por cenário,contagem de cell_id e filtro por is_modelable,CONFIRMED,CONFIRMED,8 células observadas; 7 com is_modelable=True; is_modelable=False em ['d'].,/content/drive/MyDrive/Mestrado/04-reports/99_FULL_downstream/11_FULL_model_baselines/aggregate/11_FULL_metrics_summ...,cell_id | is_modelable/status_modelagem | n_positivos_train | n_positivos_test | n_evaluable_episodes_tau_reference,contagem e filtro por célula/cenário,6f8a38786edc486fabeae9c06062fa55812801d1545384c9d77a95fadbde22ef | 7d855b1669fd7a4a34feb59c72bee8f52d29400d66fa2689f...,Unidades janela e episódio permanecem separadas.
1,4.2,"O ROC-AUC soberano das células modeláveis varia aproximadamente de 0,637 a 0,813.",M7,NB11_FULL metrics summary,min/max após seleção soberana por F1/recall/AP,CONFIRMED,CONFIRMED,ROC-AUC soberano entre 0.636979 e 0.812583.,/content/drive/MyDrive/Mestrado/04-reports/99_FULL_downstream/11_FULL_model_baselines/aggregate/11_FULL_metrics_summ...,roc_auc_tscv_mean,min/max após seleção soberana por F1/recall/AP,6f8a38786edc486fabeae9c06062fa55812801d1545384c9d77a95fadbde22ef,Faixa exclusiva das células modeláveis.
2,4.3,"No teste fixo herdado, o lift de PR-AUC sobre prevalência é calculado com numerador e denominador sobre exatamente a...",M6,escores NB11_FULL congelados + seleção primária NB14_FULL,"average_precision_score(y_true, score_calibrated) / mean(y_true)",CONFIRMED,CONFIRMED,"No teste fixo herdado, o lift de PR-AUC sobre prevalência varia de 1.546× a 5.772× nas sete células modeláveis.",/content/drive/MyDrive/Mestrado/04-reports/99_FULL_downstream/11_FULL_model_baselines/aggregate/11_FULL_scores_calib...,y_true | score_calibrated | validation=fixed_inherited_80_20 | n_scored_rows | positive_rate_tau_reference | is_prim...,"average_precision_score(y_true, score_calibrated) / mean(y_true)",6f8a38786edc486fabeae9c06062fa55812801d1545384c9d77a95fadbde22ef | cdd0f4c5a9c5c88ee44fec83e6ad8f3baa68cdf63518fca0b...,Protocolo oficial alinhado no teste fixo. O TSCV permanece como diagnóstico complementar; a antiga razão mista é ape...
3,4.4,A mediana da taxa de episódios antecipados no limiar de referência é aproximadamente 33%.,M8,"NB14_FULL: tau, numerador e denominador",mediana entre células modeláveis de antecipados/avaliáveis,CONFIRMED,CONFIRMED,Mediana entre células modeláveis = 33.33%.,/content/drive/MyDrive/Mestrado/04-reports/99_FULL_downstream/14_FULL_decision_analysis/aggregate/14_FULL_scenario_s...,tau_reference | n_evaluable_episodes_tau_reference | n_anticipated_episodes_tau_reference | episode_anticipation_rat...,mediana das taxas por célula; taxa = antecipados/avaliáveis,96485543b613c9bd1cacdc5b687ec9aab0292e94a45af597544a3eaa834dc54d,Episódios não avaliáveis não entram no denominador.
4,4.5,A Regressão Logística é o vencedor do protótipo Kaggle refinado no cenário de referência.,M7/M9,11_winner_model.json,leitura direta do campo model,CONFIRMED,CONFIRMED,Modelo vencedor do protótipo refinado: logistic_regression.,/content/drive/MyDrive/Mestrado/04-reports/11_winner_model.json,model/winner_model | roc_auc_tscv_mean,leitura direta do JSON do vencedor,f9e285963fde22029b82aa1aab209e90aefa7be37935617ccba6cf39a23587d4,Preservar protocolo e cenário na comparação.
5,4.6,"No FULL, o resultado soberano não é dominado exclusivamente pela Regressão Logística.",M4/M5,ranking NB11_FULL + governança NB13/NB14,frequência e fonte efetiva de escores,CONFIRMED,CONFIRMED,"Frequência soberana por modelo: {'hist_gradient_boosting': 4, 'logistic_regression': 3}.",/content/drive/MyDrive/Mestrado/04-reports/99_FULL_downstream/11_FULL_model_baselines/aggregate/11_FULL_metrics_summ...,model | f1_tscv_mean | recall_tscv_mean | average_precision_tscv_mean,seleção 

,check_id,check_class,description,observed,expected,status,evidence,action
0,S00_FEATURE_CONTRACT,STRUCTURAL,"Contrato vigente de atributos confirmado em fontes até NB15_FULL: raw_core=6, temporal_core=13, sem raw_7 ativo e se...","{'source_policy': {'max_full_stage': 15, 'nb16_consumed': False, 'feature_set_source': '/content/drive/MyDrive/Mestr...","{'raw_core': ['fail_rate', 'n_events', 'n_failed', 'n_machines', 'n_collections', 'event_LOST_count'], 'temporal_cor...",PASS,11_FULL_feature_sets.json | 11_FULL_model_input_features_by_cell.parquet | AUD02_feature_contract_audit.json,"Interromper a auditoria e revisar NB11_FULL se o contrato 6/13, a exclusão de event_FAIL_count ou a ausência de dupl..."
1,S00B_SEMANTIC_DUPLICATE,STRUCTURAL,Igualdade semântica entre n_failed e event_FAIL_count confirmada diretamente no Parquet model-facing FULL.,"{'rows_checked': 71424, 'mismatch_count': 0, 'semantic_duplicate_verified': True}","{'mismatch_count': 0, 'semantic_duplicate_verified': True}",PASS,"window_5min_series_allcells_model_facing_000000000000.parquet:n_failed,event_FAIL_count",Interromper a auditoria se houver qualquer divergência entre n_failed e event_FAIL_count no artefato model-facing.
2,S01,STRUCTURAL,Células observadas no NB11_FULL são a–h.,"[a, b, c, d, e, f, g, h]","[a, b, c, d, e, f, g, h]",PASS,full_metrics_summary:cell_id,"Investigar perda, duplicação ou mistura de fontes."
3,S03,STRUCTURAL,A única célula com is_modelable=False é d.,[d],[d],PASS,AUD02_modelability_summary.csv:is_modelable,Investigar o gate sem substituir o campo por rótulo.
4,S04,STRUCTURAL,A regra soberana gera um vencedor por célula modelável.,"{'a': 1, 'b': 1, 'c': 1, 'e': 1, 'f': 1, 'g': 1, 'h': 1}",1 por célula modelável,PASS,AUD02_nominal_winners_by_cell.csv,"Revisar ordenação, desempates ou duplicidades."
5,S06,STRUCTURAL,AP(score_raw) e AP(score_calibrated) preservam igualdade nos resultados soberanos.,"{'all_equal': True, 'by_cell': [{'cell_id': 'a', 'average_precision_score_raw': 0.31214320936074474, 'average_precis...","{'all_equal': True, 'interpretation': 'resultado observado; não pressuposto geral sobre qualquer método de calibração'}",PASS,AUD02_pr_auc_lift_fixed_test_by_cell.csv,"Se houver divergência, verificar mudança de ordenação pela calibração; o lift oficial continua baseado no score_cali..."
6,R02,REGRESSION,Quantidade de células modeláveis versus snapshot.,7,7,PASS,AUD02_nominal_winners_by_cell.csv,Investigar; atualizar snapshot só com justificativa.
7,R05,REGRESSION,Frequência empírica dos modelos soberanos.,"{'hist_gradient_boosting': 4, 'logistic_regression': 3}","{'hist_gradient_boosting': 4, 'logistic_regression': 3}",PASS,AUD02_model_frequency.csv,Mudança não implica erro automático.
8,R06,REGRESSION,ROC-AUC mínimo soberano.,0.636979,0.636979,PASS,AUD02_kaggle_comparison.csv,"Investigar vencedor, cenário ou métrica."
9,R07,REGRESSION,ROC-AUC máximo soberano.,0.812583,0.812583,PASS,AUD02_kaggle_comparison.csv,"Investigar vencedor, cenário ou métrica."


AUD02 CONCLUÍDO
Status geral : PASS_WITH_REGRESSION_REVIEW
Saída        : /content/drive/MyDrive/Mestrado/04-reports/AUD02_evidence_audit
Manifesto    : /content/drive/MyDrive/Mestrado/04-reports/AUD02_evidence_audit/AUD02_artifact_manifest_sha256.csv
NB16 lido?   : False
Próxima etapa: após aprovação do AUD02, executar DOC01; depois atualizar seletivamente o Mapa da Escrita. NB16_FULL permanece preservado e não foi consumido.


# AUD02 — Conclusão da etapa

**Status automático da auditoria:** `PASS_WITH_REGRESSION_REVIEW`  
**Aceite do Grupo 3:** **APROVADO**, com uma revisão documental não bloqueante explicitada no item 5.

## 1. Escopo e integridade da execução

A auditoria foi concluída em modo somente-leitura sobre a linhagem experimental vigente. Foram registradas **16 fontes**, **9 afirmações auditadas** e **19 checks**, sem consumo do NB16_FULL e sem treinamento, reotimização ou alteração dos artefatos experimentais.

O fechamento confirmou **0 falhas estruturais**, **0 falhas de rastreabilidade** e **0 qualificações metodológicas pendentes**. O inventário de fontes contém 16 entradas únicas, todas com SHA-256 registrado. Entre as fontes com manifesto reconhecido pela convenção do AUD02, **13/13 apresentaram `MATCH`**. As três fontes sem manifesto localizado pela convenção de nomes permaneceram rastreadas por hash próprio e não produziram falha de integridade.

Os schemas totalizaram **61 verificações obrigatórias/aplicáveis com `PASS`** e **6 campos opcionais ausentes**, sem ausência de campo obrigatório.

## 2. Contrato de atributos e redundância

O contrato de atributos da linhagem foi confirmado diretamente em artefatos anteriores ao fechamento:

- `raw_core`: **6 atributos**;
- `temporal_core`: **13 atributos**;
- `raw_7`: ausente como feature set ativo;
- `event_FAIL_count`: ausente dos preditores e da matriz efetivamente entregue aos modelos;
- `n_failed == event_FAIL_count`: confirmado nas **71.424 linhas** do Parquet model-facing FULL, com **0 divergências**;
- atributos esperados de `temporal_core`: todos presentes na matriz de entrada do NB11_FULL;
- duplicidades exatas injustificadas entre os atributos ativos: **nenhuma**.

Assim, a coluna `event_FAIL_count` permanece apenas como elemento de proveniência, enquanto `n_failed` constitui a representação utilizada no vetor preditor.

## 3. Resultados auditados

Foram observadas as oito células `a`–`h`. Sete permanecem modeláveis (`a`, `b`, `c`, `e`, `f`, `g`, `h`) e a célula `d` permanece fora da comparação soberana por não satisfazer o gate de modelabilidade.

Entre as células modeláveis:

- `hist_gradient_boosting` é soberano em **4 células** e `logistic_regression` em **3**;
- o ROC-AUC soberano varia de **0,636979 a 0,812583**;
- **6 das 7** células modeláveis superam descritivamente a referência de **0,647387**; a célula `f` fica abaixo;
- a mediana da taxa de episódios antecipados no limiar de referência é **33,33%**;
- o lift oficial de PR-AUC sobre prevalência, calculado no **teste fixo herdado** com numerador e denominador nas mesmas observações, varia de **1,546× a 5,772×**;
- `AP(score_raw) = AP(score_calibrated)` nas sete células soberanas.

A política por métrica permanece separada: TSCV para seleção/discriminação e comparação descritiva de ROC-AUC; teste fixo herdado para lift e métricas operacionais.

## 4. Governança da LSTM, checks e artefatos

Nenhuma LSTM foi promovida como fonte primária. Na linhagem auditada, apenas a célula `b` ultrapassa o piso nominal `ΔF1 >= 0,03`; nenhuma célula atinge o gate de robustez `ΔF1/SE >= 2,0`. As sete células modeláveis mantêm a LSTM somente como avaliação secundária no NB14_FULL.

Dos **19 checks**, **18 resultaram em `PASS`** e **1 em `REVIEW`**. O manifesto final registra **27 artefatos próprios do AUD02**. Na conferência externa do Grupo 3, os hashes de oito artefatos centrais — `summary`, `checks`, auditoria do contrato de atributos, inventário e hashes das fontes, checks de schema, notas de conclusão e relatório de afirmações — foram recalculados e coincidiram com o manifesto em **8/8 casos**.

As nove afirmações auditadas ficaram coerentes com os artefatos atuais: os itens 4.1–4.7 e 4.9 foram confirmados; o item 4.8 permanece deliberadamente como contexto metodológico sobre Brier/calibração.

## 5. Revisão documental R10

O único `REVIEW` corresponde ao check **R10 — “Células ausentes da comparação”**. O observado é `['d']`, enquanto o snapshot inicial registrou lista vazia.

A revisão é **não bloqueante**: a ausência da célula `d` decorre diretamente do gate de modelabilidade já confirmado pelos checks estruturais e pela afirmação 4.1. A célula `d` não deve integrar a comparação soberana do item 4.9. Portanto, não há inconsistência científica nos resultados nem necessidade de reexecução experimental.

O status `PASS_WITH_REGRESSION_REVIEW` é preservado para manter a rastreabilidade do snapshot inicial, sem converter essa revisão documental em falha.

## 6. Fechamento e próxima etapa

O AUD02 está **apto para sustentar a consolidação documental da linhagem corrigida**. A auditoria confirma o contrato 6/13, a exclusão da redundância dos preditores, a rastreabilidade dos resultados por célula e a separação dos protocolos de avaliação.

A próxima etapa é executar o **DOC01** para regenerar a Matriz de Notebooks a partir da base semântica curada e dos artefatos vigentes, seguida da atualização seletiva do Mapa da Escrita e dos demais instrumentos internos de controle.
